In [ ]:
from pathlib import Path
from IPython.display import Image, display

cover_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/S6E5_Driver_01.png")
if cover_image_path.exists():
    display(Image(filename=str(cover_image_path)))

# 🏎️💥 Driver's High

**Climbing the CV ladder with Driver signal.** This notebook tests a narrow claim: `Driver` can add a small, measurable marginal signal only when it survives the same OOF validation protocol as every other feature block.

The first half asks whether raw Driver identity, compressed Driver statistics, and Driver interactions improve the race-state baseline. The second half adds the newer ideas: public-stack preprocessing candidates and logit-space Driver residual correction.

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-left:6px solid #B5473C; border-radius:14px; padding:14px 16px; margin:14px 0 18px 0; color:#233127; line-height:1.55; box-shadow:0 8px 18px rgba(35,49,39,0.04);">
  <div style="font-size:13px; font-weight:900; letter-spacing:1.8px; color:#7A6045; text-transform:uppercase; margin-bottom:7px;">AUC Delta Rule</div>
  <div style="font-size:20px; font-weight:900; color:#233127; margin-bottom:8px;">Delta = AUC(candidate) - AUC(counterfactual)</div>
  <table style="width:100%; border-collapse:collapse; font-size:13.5px; line-height:1.42;">
    <thead><tr style="background:#233127; color:#FFFCF8; text-align:left;"><th style="padding:8px 10px; border:1px solid #D9D0BF;">Test type</th><th style="padding:8px 10px; border:1px solid #D9D0BF;">Counterfactual</th><th style="padding:8px 10px; border:1px solid #D9D0BF;">Read</th></tr></thead>
    <tbody>
      <tr><td style="padding:8px 10px; border:1px solid #E2D8C7; font-weight:900; color:#2F6B4F;">Sequential ladder</td><td style="padding:8px 10px; border:1px solid #E2D8C7;">previous step</td><td style="padding:8px 10px; border:1px solid #E2D8C7;">positive delta becomes evidence candidate</td></tr>
      <tr style="background:#F7F3EA;"><td style="padding:8px 10px; border:1px solid #E2D8C7; font-weight:900; color:#3D6F8A;">Interaction branch</td><td style="padding:8px 10px; border:1px solid #E2D8C7;">B04 parent</td><td style="padding:8px 10px; border:1px solid #E2D8C7;">reject if it only beats the previous row</td></tr>
      <tr><td style="padding:8px 10px; border:1px solid #E2D8C7; font-weight:900; color:#C7772C;">Residual layer</td><td style="padding:8px 10px; border:1px solid #E2D8C7;">uncorrected base</td><td style="padding:8px 10px; border:1px solid #E2D8C7;">promote only if it clears the AUC threshold</td></tr>
      <tr style="background:#F7F3EA;"><td style="padding:8px 10px; border:1px solid #E2D8C7; font-weight:900; color:#B5473C;">Heavy-stack overlay</td><td style="padding:8px 10px; border:1px solid #E2D8C7;">best stack artifact</td><td style="padding:8px 10px; border:1px solid #E2D8C7;">Driver +alpha must beat the real scoring base</td></tr>
    </tbody>
  </table>
</div>


In [ ]:
from IPython.display import HTML, display

display(HTML("""
<div style="max-width:1160px; margin:8px 0 24px; padding:22px; border-radius:22px; border:1px solid #D9D0BF; background:linear-gradient(135deg,#F7F3EA 0%,#EEF4ED 100%); color:#233127; box-shadow:0 12px 30px rgba(35,49,39,0.08); font-family:Inter,Arial,sans-serif;">
  <div style="display:flex; flex-wrap:wrap; align-items:flex-start; justify-content:space-between; gap:16px; margin-bottom:18px;">
    <div style="min-width:280px; flex:1 1 600px;">
      <div style="font-size:12px; font-weight:900; letter-spacing:2.6px; color:#7A6045; text-transform:uppercase; margin-bottom:8px;">Driver Feature Argument</div>
      <div style="font-size:31px; line-height:1.12; font-weight:900; color:#233127; margin-bottom:8px;">Can Driver survive the same OOF test as the race-state features?</div>
      <div style="font-size:15px; line-height:1.55; font-weight:700; color:#4C5A4C; max-width:780px;">The notebook does not start by declaring Driver important or unimportant. It fixes the race/tyre foundation, changes one Driver representation at a time, then checks whether stack-style preprocessing or Driver residual correction creates additional lift.</div>
    </div>
    <div style="flex:0 1 310px; background:#FFFCF8; border:1px solid #D9D0BF; border-radius:16px; padding:14px 16px;">
      <div style="font-size:12px; font-weight:900; letter-spacing:2px; color:#6F7C69; text-transform:uppercase; margin-bottom:7px;">Validation Rule</div>
      <div style="font-size:20px; font-weight:900; color:#233127; margin-bottom:5px;">Same splits, one change</div>
      <div style="font-size:13px; line-height:1.45; font-weight:700; color:#5F6B60;">A Driver block is promoted only when it beats its intended counterfactual under the same OOF protocol.</div>
    </div>
  </div>

  <div style="background:#FFFCF8; border:1px solid #D9D0BF; border-radius:18px; padding:16px 18px; margin:14px auto 20px; box-shadow:0 8px 18px rgba(35,49,39,0.06);">
    <div style="font-size:12px; font-weight:900; letter-spacing:2px; color:#6F7C69; text-transform:uppercase; margin-bottom:12px;">Shared Foundation</div>
    <div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(170px,1fr)); gap:10px;">
      <div style="background:#EEF4ED; border:1px solid #B8C3B4; border-radius:13px; padding:10px 11px;"><div style="font-size:14px; font-weight:900; color:#2F6B4F;">race / tyre algebra</div><div style="font-size:12px; color:#5F6B60; font-weight:700; line-height:1.4;">relative lap, tyre age, remaining race</div></div>
      <div style="background:#FFF4DE; border:1px solid #D9C69F; border-radius:13px; padding:10px 11px;"><div style="font-size:14px; font-weight:900; color:#7A6045;">compound pressure</div><div style="font-size:12px; color:#5F6B60; font-weight:700; line-height:1.4;">expected tyre life and pit window</div></div>
      <div style="background:#FCEDEA; border:1px solid #E2B7B2; border-radius:13px; padding:10px 11px;"><div style="font-size:14px; font-weight:900; color:#B5473C;">state interactions</div><div style="font-size:12px; color:#5F6B60; font-weight:700; line-height:1.4;">tyre x stint x race phase</div></div>
      <div style="background:#EFF7FB; border:1px solid #B7CEDA; border-radius:13px; padding:10px 11px;"><div style="font-size:14px; font-weight:900; color:#3D6F8A;">fold safety</div><div style="font-size:12px; color:#5F6B60; font-weight:700; line-height:1.4;">OOF encodings do not read their own labels</div></div>
    </div>
  </div>

  <div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(260px,1fr)); gap:14px;">
    <div style="background:#F7F3EA; border:1px solid #E2D8C7; border-radius:18px; padding:15px 16px;">
      <div style="font-size:12px; font-weight:900; letter-spacing:2px; color:#2F6B4F; text-transform:uppercase; margin-bottom:8px;">Front Half</div>
      <div style="font-size:22px; line-height:1.2; font-weight:900; margin-bottom:9px;">Driver ladder</div>
      <div style="font-size:13px; line-height:1.5; color:#4C5A4C; font-weight:700; margin-bottom:11px;">No Driver → raw Driver OHE → string/vocab/frequency → target encoding → interaction target encoding.</div>
      <div style="display:flex; flex-wrap:wrap; gap:7px;"><span style="background:#233127;color:#FFFCF8;border-radius:999px;padding:7px 11px;font-size:12px;font-weight:900;">A00/B00</span><span style="background:#2F6B4F;color:#FFFCF8;border-radius:999px;padding:7px 11px;font-size:12px;font-weight:900;">OHE</span><span style="background:#3D6F8A;color:#FFFCF8;border-radius:999px;padding:7px 11px;font-size:12px;font-weight:900;">freq</span><span style="background:#C7772C;color:#FFFCF8;border-radius:999px;padding:7px 11px;font-size:12px;font-weight:900;">TE</span></div>
    </div>
    <div style="background:#FFFCF8; border:1px solid #E2D8C7; border-radius:18px; padding:15px 16px;">
      <div style="font-size:12px; font-weight:900; letter-spacing:2px; color:#3D6F8A; text-transform:uppercase; margin-bottom:8px;">New Candidate</div>
      <div style="font-size:22px; line-height:1.2; font-weight:900; margin-bottom:9px;">Stack preprocessing</div>
      <div style="font-size:13px; line-height:1.5; color:#4C5A4C; font-weight:700; margin-bottom:11px;">Bring in the high-score-stack feature matrices as candidates: compact core preprocessing and domain-rich sequence/group preprocessing.</div>
      <div style="display:flex; flex-wrap:wrap; gap:7px;"><span style="background:#3D6F8A;color:#FFFCF8;border-radius:999px;padding:7px 11px;font-size:12px;font-weight:900;">S05 core</span><span style="background:#B5473C;color:#FFFCF8;border-radius:999px;padding:7px 11px;font-size:12px;font-weight:900;">S06 domain</span></div>
    </div>
    <div style="background:#FFF4DE; border:1px solid #D9C69F; border-radius:18px; padding:15px 16px;">
      <div style="font-size:12px; font-weight:900; letter-spacing:2px; color:#7A6045; text-transform:uppercase; margin-bottom:8px;">Back Half</div>
      <div style="font-size:22px; line-height:1.2; font-weight:900; margin-bottom:9px;">Residual correction</div>
      <div style="font-size:13px; line-height:1.5; color:#4C5A4C; font-weight:700; margin-bottom:11px;">If Driver is too small as a main feature, test whether Driver group bias can adjust the base model logit.</div>
      <div style="display:flex; flex-wrap:wrap; gap:7px;"><span style="background:#C7772C;color:#FFFCF8;border-radius:999px;padding:7px 11px;font-size:12px;font-weight:900;">logit bias</span><span style="background:#233127;color:#FFFCF8;border-radius:999px;padding:7px 11px;font-size:12px;font-weight:900;">promote only by OOF</span></div>
    </div>
  </div>
</div>
"""))


## 0. Executive Map

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-left:7px solid #B5473C; border-radius:16px; padding:17px 19px; margin:12px 0 20px 0; color:#233127; box-shadow:0 10px 22px rgba(35,49,39,0.05);">
  <div style="font-size:13px; font-weight:900; letter-spacing:2px; color:#7A6045; text-transform:uppercase; margin-bottom:8px;">What this notebook tries to prove</div>
  <div style="font-size:1.38rem; font-weight:900; line-height:1.25; color:#233127; margin-bottom:12px;">Driver signal is allowed to be small. The question is whether any Driver representation survives a fair counterfactual test.</div>
  <table style="width:100%; border-collapse:collapse; font-size:13.5px; line-height:1.45;">
    <thead><tr style="background:#233127; color:#FFFCF8; text-align:left;"><th style="padding:10px 11px; border:1px solid #D9D0BF; width:24%;">Stage</th><th style="padding:10px 11px; border:1px solid #D9D0BF; width:30%;">Why it exists</th><th style="padding:10px 11px; border:1px solid #D9D0BF; width:22%;">What would count as evidence?</th><th style="padding:10px 11px; border:1px solid #D9D0BF; width:24%;">How to read failures</th></tr></thead>
    <tbody>
      <tr><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#2F6B4F;">Race-state floor</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Pit timing should mostly come from lap, tyre, compound, stint, pace, and degradation.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Strong no-Driver OOF baseline.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">This is the benchmark Driver must beat.</td></tr>
      <tr style="background:#F7F3EA;"><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#3D6F8A;">Driver ladder</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">High-cardinality IDs might encode repeated strategy context or synthetic priors.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Positive OOF delta versus the matching no-Driver or parent step.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Rejected Driver blocks still teach us which shortcut is unstable.</td></tr>
      <tr><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#B5473C;">Stack preprocessing</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Public high-score notebooks use ordinal/frequency matrices and sequence/group features.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">S05/S06 beat the selected Driver ladder result.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">If they fail here, they may still belong in the heavier XGB/Cat/original stack.</td></tr>
      <tr style="background:#F7F3EA;"><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#C7772C;">Residual correction</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Driver might be too small as a main feature but useful as post-stack bias correction.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Logit-corrected OOF AUC clears the promotion threshold.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Tiny gains stay on watchlist, not in the final blend.</td></tr>
    </tbody>
  </table>
</div>


## 0.1 Logic Map

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-left:7px solid #3D6F8A; border-radius:16px; padding:17px 19px; margin:12px 0 20px 0; color:#233127; box-shadow:0 10px 22px rgba(35,49,39,0.05);">
  <div style="font-size:13px; font-weight:900; letter-spacing:2px; color:#3D6F8A; text-transform:uppercase; margin-bottom:7px;">How the proof works</div>
  <div style="font-size:1.28rem; font-weight:900; line-height:1.28; color:#233127; margin-bottom:12px;">Every block has a counterfactual. No counterfactual win, no promotion.</div>
  <div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(215px,1fr)); gap:11px; margin-bottom:15px;">
    <div style="background:#F7F3EA; border:1px solid #E2D8C7; border-radius:13px; padding:12px 13px;"><div style="font-size:12px; font-weight:900; color:#2F6B4F; text-transform:uppercase; letter-spacing:.08em; margin-bottom:5px;">A. Floor</div><div style="font-weight:900; margin-bottom:4px;">Race-state without Driver</div><div style="font-size:12.5px; color:#5F6B60; line-height:1.5;">Build a strong non-Driver baseline before giving Driver any credit.</div></div>
    <div style="background:#F7F3EA; border:1px solid #E2D8C7; border-radius:13px; padding:12px 13px;"><div style="font-size:12px; font-weight:900; color:#C7772C; text-transform:uppercase; letter-spacing:.08em; margin-bottom:5px;">B. Identity</div><div style="font-weight:900; margin-bottom:4px;">Raw Driver OHE</div><div style="font-size:12.5px; color:#5F6B60; line-height:1.5;">Tests whether the ID itself has direct ranking value.</div></div>
    <div style="background:#F7F3EA; border:1px solid #E2D8C7; border-radius:13px; padding:12px 13px;"><div style="font-size:12px; font-weight:900; color:#3D6F8A; text-transform:uppercase; letter-spacing:.08em; margin-bottom:5px;">C. Compression</div><div style="font-weight:900; margin-bottom:4px;">String, vocab, frequency, TE</div><div style="font-size:12.5px; color:#5F6B60; line-height:1.5;">Keeps only the Driver pieces that improve OOF ranking.</div></div>
    <div style="background:#F7F3EA; border:1px solid #E2D8C7; border-radius:13px; padding:12px 13px;"><div style="font-size:12px; font-weight:900; color:#B5473C; text-transform:uppercase; letter-spacing:.08em; margin-bottom:5px;">D. Residual</div><div style="font-weight:900; margin-bottom:4px;">Logit-space correction</div><div style="font-size:12.5px; color:#5F6B60; line-height:1.5;">Driver can only adjust the base logit if OOF delta clears the promotion threshold.</div></div>
  </div>
  <table style="width:100%; border-collapse:collapse; font-size:13.5px; line-height:1.48;">
    <thead><tr style="background:#233127; color:#FFFCF8; text-align:left;"><th style="padding:10px 11px; border:1px solid #D9D0BF; width:19%;">Question</th><th style="padding:10px 11px; border:1px solid #D9D0BF; width:26%;">Counterfactual</th><th style="padding:10px 11px; border:1px solid #D9D0BF; width:22%;">Statistic</th><th style="padding:10px 11px; border:1px solid #D9D0BF; width:33%;">Action</th></tr></thead>
    <tbody>
      <tr><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#C7772C;">Does raw ID help?</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">A01 Driver OHE vs A00 no Driver</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">A01 AUC - A00 AUC</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Keep as a small identity prior only if OOF lift is positive.</td></tr>
      <tr style="background:#F7F3EA;"><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#2F6B4F;">Can Driver be compressed?</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">B01-B04 vs B00 and cumulative best</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Cumulative-best OOF AUC</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Promote the simplest representation that improves OOF.</td></tr>
      <tr><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#B5473C;">Are interactions stable?</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">B05-B08 each vs B04 parent</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Delta vs parent</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Accept only if the interaction beats its own parent, not just the previous row.</td></tr>
      <tr style="background:#F7F3EA;"><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#3D6F8A;">Can Driver add residual +alpha?</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Base logit vs base logit + Driver logit bias</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Corrected AUC - base AUC</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Promote only if the gain clears the conservative threshold.</td></tr>
      <tr><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#7A6045;">Where is score upside?</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Driver stack vs public heavy GBDT references</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">OOF gap</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Move Driver into auxiliary role beside original-data-heavy ensembles.</td></tr>
    </tbody>
  </table>
</div>


## Table of Contents

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-radius:16px; padding:18px 20px; margin:12px 0 20px 0; box-shadow:0 8px 20px rgba(35,49,39,0.04); color:#233127;">
  <div style="font-size:1.35rem; font-weight:900; margin-bottom:6px; color:#233127;">Notebook Flow</div>
  <div style="font-size:13px; color:#5F6B60; line-height:1.55; margin-bottom:14px;">The old Driver ladder remains the spine. The additions are placed after it as score-oriented candidates: stack preprocessing, artifact-based heavy-stack residual overlay, and Driver residual correction.</div>
  <div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(245px,1fr)); gap:12px;">
    <div style="border:1px solid #E4DAC9; border-radius:14px; padding:13px 14px; background:#F7F3EA;"><div style="font-size:12px; text-transform:uppercase; letter-spacing:.08em; color:#6F7C69; font-weight:900; margin-bottom:6px;">Setup</div><div style="font-weight:900; color:#2F6B4F; margin-bottom:6px;">0-2. Map and Inventory</div><div style="font-size:12.5px; line-height:1.55; color:#233127;">Question, counterfactuals, and Driver cardinality.</div></div>
    <div style="border:1px solid #E4DAC9; border-radius:14px; padding:13px 14px; background:#F7F3EA;"><div style="font-size:12px; text-transform:uppercase; letter-spacing:.08em; color:#6F7C69; font-weight:900; margin-bottom:6px;">Foundation</div><div style="font-weight:900; color:#3D6F8A; margin-bottom:6px;">3-5. Features and Baseline</div><div style="font-size:12.5px; line-height:1.55; color:#233127;">Race-state blocks, fold-safe encoders, transparent no-Driver floor.</div></div>
    <div style="border:1px solid #E4DAC9; border-radius:14px; padding:13px 14px; background:#F7F3EA;"><div style="font-size:12px; text-transform:uppercase; letter-spacing:.08em; color:#6F7C69; font-weight:900; margin-bottom:6px;">Driver Evidence</div><div style="font-weight:900; color:#C7772C; margin-bottom:6px;">6. Driver Ladder</div><div style="font-size:12.5px; line-height:1.55; color:#233127;">Raw ID, compressed Driver features, target encoding, and interaction TE.</div></div>
    <div style="border:1px solid #E4DAC9; border-radius:14px; padding:13px 14px; background:#F7F3EA;"><div style="font-size:12px; text-transform:uppercase; letter-spacing:.08em; color:#6F7C69; font-weight:900; margin-bottom:6px;">Score Candidates</div><div style="font-weight:900; color:#B5473C; margin-bottom:6px;">7-14. Stack, Residuals, Submission</div><div style="font-size:12.5px; line-height:1.55; color:#233127;">Public-domain candidates, stack preprocessing, F1 diagnostic, local and artifact-based residual correction, optional model stack, final file.</div></div>
  </div>
</div>


## 1. Configuration, Imports, and Display Helpers


In [ ]:
from __future__ import annotations

import os
import re
import warnings
from dataclasses import dataclass
from html import escape
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
except Exception:
    lgb = None

try:
    import xgboost as xgb
except Exception:
    xgb = None

try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None

RANDOM_STATE = 42
FAST_MODE = os.environ.get("S6E5_FAST_MODE", "0") == "1"
USE_GPU = os.environ.get("S6E5_USE_GPU", "0") == "1"

N_SPLITS = 2 if FAST_MODE else int(os.environ.get("S6E5_N_SPLITS", "3"))
INNER_TE_SPLITS = 2 if FAST_MODE else int(os.environ.get("S6E5_INNER_TE_SPLITS", "5"))
DEFAULT_ROW_LIMIT = 60_000 if FAST_MODE else int(os.environ.get("S6E5_ROW_LIMIT", "180000"))
CV_ROW_LIMIT = None if DEFAULT_ROW_LIMIT <= 0 else DEFAULT_ROW_LIMIT

if os.environ.get("S6E5_LADDER_MODEL") is not None:
    LADDER_MODEL = os.environ["S6E5_LADDER_MODEL"]
elif lgb is not None:
    LADDER_MODEL = "lgbm"
elif xgb is not None:
    LADDER_MODEL = "xgb"
else:
    LADDER_MODEL = "logreg"

RUN_FULL_FEATURE_LADDER = os.environ.get("S6E5_RUN_FULL_LADDER", "0") == "1"
RUN_ORIGINAL_ABLATION = os.environ.get("S6E5_RUN_ORIGINAL_ABLATION", "0") == "1"
RUN_MODEL_UPGRADE = os.environ.get("S6E5_RUN_MODEL_UPGRADE", "0") == "1"
RUN_SCORE_STACK = os.environ.get("S6E5_RUN_SCORE_STACK", "1") == "1"
RUN_RESIDUAL_CORRECTION = os.environ.get("S6E5_RUN_RESIDUAL_CORRECTION", "1") == "1"
RUN_HEAVY_STACK_RESIDUAL = os.environ.get("S6E5_RUN_HEAVY_STACK_RESIDUAL", "1") == "1"
STACK_BLEND_PROFILE = os.environ.get("S6E5_BLEND_PROFILE", "fast" if FAST_MODE else "full").lower()
STACK_BLEND_GRID_STEP = float(os.environ.get("S6E5_BLEND_GRID_STEP", "0.10" if FAST_MODE else "0.05"))
STACKING_INPUT_PATH = Path(os.environ.get("S6E5_STACKING_INPUT", "/kaggle/input/notebooks/pilkwang/s6e5-high-score-stacking"))
REALMLP_INPUT_PATH = Path(os.environ.get("S6E5_REALMLP_INPUT", "/kaggle/input/notebooks/pilkwang/s6e5-realmlp-oof-artifact-builder"))
EXTRA_ARTIFACT_INPUTS = [Path(path.strip()) for path in os.environ.get("S6E5_EXTRA_ARTIFACT_INPUTS", "").split(",") if path.strip()]

ORIGINAL_WEIGHTS = [0.0, 0.25] if FAST_MODE else [0.0, 0.10, 0.25, 0.50]
FINAL_MODEL_NAMES = ["xgb", "lgbm", "cat"] if USE_GPU else ["lgbm", "xgb", "cat"]
LGBM_N_ESTIMATORS = int(os.environ.get("S6E5_LGBM_ESTIMATORS", "180" if FAST_MODE else "3000"))
XGB_N_ESTIMATORS = int(os.environ.get("S6E5_XGB_ESTIMATORS", "180" if FAST_MODE else "4000"))
CAT_N_ITERATIONS = int(os.environ.get("S6E5_CAT_ITERATIONS", "180" if FAST_MODE else "3000"))
SUBMISSION_FILENAME = "submission.csv"
TARGET_CANDIDATES = ["PitNextLap", "PitNextLab"]

COMPETITION_PATHS = [
    Path("/kaggle/input/competitions/playground-series-s6e5"),
    Path("/kaggle/input/playground-series-s6e5"),
    Path("."),
]
ORIGINAL_PATHS = [
    Path("/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction"),
    Path("/kaggle/input/f1-strategy-dataset-pit-stop-prediction"),
    Path("."),
]

COLORS = {
    "ink": "#213027",
    "muted": "#6f7b6d",
    "paper": "#faf7ef",
    "line": "#ded4c0",
    "green": "#2f6f4e",
    "blue": "#3f7f9f",
    "gold": "#c77c2b",
    "red": "#c54d43",
}

sns.set_theme(
    style="whitegrid",
    rc={
        "figure.facecolor": COLORS["paper"],
        "axes.facecolor": "#fffdf8",
        "axes.edgecolor": COLORS["line"],
        "grid.color": "#ece4d5",
        "text.color": COLORS["ink"],
        "axes.labelcolor": COLORS["ink"],
        "xtick.color": COLORS["muted"],
        "ytick.color": COLORS["muted"],
    },
)


CV_PRECISION_COLUMNS = {
    "OOF_AUC", "OOF_LogLoss", "Delta_vs_Previous", "Delta_vs_Baseline",
    "Delta_vs_No_Driver", "Delta_vs_Parent", "Delta_vs_Base", "Delta", "Cumulative_Best_AUC",
    "Mean_Delta", "P05", "Median", "P95", "Prob_Positive", "Blend_Weight",
    "Reference_OOF_AUC", "Gap_vs_Current_Driver", "Delta_vs_Selected_Driver",
}


def _fmt_value(value, precision=4):
    if pd.isna(value):
        return ""
    if isinstance(value, (np.integer, int)):
        return f"{int(value):,}"
    if isinstance(value, (np.floating, float)):
        if abs(value) >= 1000:
            return f"{value:,.1f}"
        return f"{value:.{precision}f}" if abs(value) < 1 else f"{value:.3f}"
    return str(value)


def display_table(frame, title=None, percent_cols=None, precision=4, max_rows=30):
    shown = frame.copy()
    if max_rows is not None:
        shown = shown.head(max_rows)
    percent_cols = set(percent_cols or [])
    for col in shown.columns:
        if col in percent_cols:
            shown[col] = shown[col].map(lambda x: "" if pd.isna(x) else f"{100 * x:.2f}%")
        elif pd.api.types.is_numeric_dtype(shown[col]):
            col_precision = 6 if col in CV_PRECISION_COLUMNS else precision
            shown[col] = shown[col].map(lambda x, p=col_precision: _fmt_value(x, precision=p))
    title_html = f"<div class='tbl-title'>{escape(title)}</div>" if title else ""
    html = f"""
    <style>
      .tbl-wrap {{border:1px solid {COLORS['line']}; border-radius:14px; overflow:hidden; margin:12px 0 22px; background:{COLORS['paper']};}}
      .tbl-title {{font-size:22px; font-weight:800; padding:16px 20px; color:{COLORS['ink']};}}
      .tbl-wrap table {{width:100%; border-collapse:collapse; font-size:14px;}}
      .tbl-wrap th {{background:{COLORS['ink']}; color:#fffaf0; text-align:left; padding:10px 12px;}}
      .tbl-wrap td {{padding:9px 12px; border-top:1px solid #eadfcb; color:{COLORS['ink']};}}
      .tbl-wrap tr:nth-child(even) td {{background:#f5f0e7;}}
      .pill {{display:inline-block; padding:3px 9px; border-radius:999px; font-weight:800; border:1px solid {COLORS['line']};}}
    </style>
    <div class='tbl-wrap'>{title_html}{shown.to_html(index=False, escape=False)}</div>
    """
    display(HTML(html))


def metric_card(title, value, subtitle="", color=None):
    color = color or COLORS["green"]
    value_text = str(value)
    value_size = "23px" if len(value_text) <= 12 else "21px"
    return f"""
    <div style='border:1px solid {COLORS['line']}; border-left:7px solid {color}; border-radius:14px; padding:13px 15px; background:#fffdf8; min-height:128px; box-sizing:border-box;'>
      <div style='font-size:12px; line-height:1.25; color:{COLORS['muted']}; font-weight:900; letter-spacing:.06em; text-transform:uppercase; min-height:31px;'>{escape(title)}</div>
      <div style='font-size:{value_size}; line-height:1.14; color:{COLORS['ink']}; font-weight:900; margin-top:9px; word-break:normal;'>{escape(value_text)}</div>
      <div style='font-size:12.5px; line-height:1.32; color:{COLORS['muted']}; margin-top:9px;'>{escape(subtitle)}</div>
    </div>
    """


def display_cards(cards, columns=4):
    html = f"<div style='display:grid; grid-template-columns:repeat({columns}, minmax(0, 1fr)); gap:12px; margin:12px 0 22px;'>" + "".join(cards) + "</div>"
    display(HTML(html))


def resolve_file(filename, roots, required=True):
    for root in roots:
        candidate = root / filename
        if candidate.exists():
            return candidate
    if required:
        raise FileNotFoundError(f"Could not find {filename} in: {roots}")
    return None


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True, dtype=np.float32)

runtime_summary = pd.DataFrame({
    "Setting": [
        "FAST_MODE", "USE_GPU", "N_SPLITS", "INNER_TE_SPLITS", "CV_ROW_LIMIT",
        "LADDER_MODEL", "RUN_SCORE_STACK", "RUN_RESIDUAL_CORRECTION", "RUN_HEAVY_STACK_RESIDUAL",
        "STACK_BLEND_PROFILE", "RUN_FULL_FEATURE_LADDER", "RUN_ORIGINAL_ABLATION", "RUN_MODEL_UPGRADE",
    ],
    "Value": [
        FAST_MODE, USE_GPU, N_SPLITS, INNER_TE_SPLITS,
        "full train" if CV_ROW_LIMIT is None else f"{CV_ROW_LIMIT:,} rows",
        LADDER_MODEL, RUN_SCORE_STACK, RUN_RESIDUAL_CORRECTION, RUN_HEAVY_STACK_RESIDUAL,
        STACK_BLEND_PROFILE, RUN_FULL_FEATURE_LADDER, RUN_ORIGINAL_ABLATION, RUN_MODEL_UPGRADE,
    ],
})
display_table(runtime_summary, "Runtime configuration for this saved output")

runtime_note = "Full training rows are used." if CV_ROW_LIMIT is None else "This saved output is a capped CV run; use S6E5_ROW_LIMIT=0 for full-train confirmation."
display(HTML(f"""
<div style='background:#FFFCF8; border:1px solid {COLORS['line']}; border-left:7px solid {COLORS['gold']}; border-radius:14px; padding:13px 15px; margin:8px 0 22px; color:{COLORS['ink']}; line-height:1.55;'>
  <div style='font-size:12px; font-weight:900; letter-spacing:.1em; color:#7A6045; text-transform:uppercase; margin-bottom:5px;'>Runtime Read</div>
  <div style='font-weight:900; margin-bottom:4px;'>Saved scores should be interpreted with this profile.</div>
  <div style='font-size:13px; color:{COLORS['muted']};'>{runtime_note} Small Driver deltas should be confirmed with more folds and uncapped rows before treating them as final leaderboard evidence.</div>
</div>
"""))



## 2. Data Loading and Driver Inventory


In [ ]:
train_path = resolve_file("train.csv", COMPETITION_PATHS)
test_path = resolve_file("test.csv", COMPETITION_PATHS)
submission_path = resolve_file("sample_submission.csv", COMPETITION_PATHS)
original_path = resolve_file("f1_strategy_dataset_v4.csv", ORIGINAL_PATHS, required=False)

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission_template = pd.read_csv(submission_path)
original = pd.read_csv(original_path) if original_path is not None else None

def resolve_target_column(df, candidates, label):
    target = next((col for col in candidates if col in df.columns), None)
    if target is None:
        raise ValueError(f"No target column found in {label} among {candidates}")
    return target

TARGET = resolve_target_column(train, TARGET_CANDIDATES, "train")
SUBMISSION_TARGET = resolve_target_column(submission_template, TARGET_CANDIDATES, "sample_submission")

if original is not None:
    ORIGINAL_TARGET = resolve_target_column(original, TARGET_CANDIDATES, "original")
    if ORIGINAL_TARGET != TARGET:
        original = original.rename(columns={ORIGINAL_TARGET: TARGET})

ID_COL = "id"
base_features = [col for col in test.columns if col != ID_COL]
raw_numeric_cols = [col for col in base_features if pd.api.types.is_numeric_dtype(train[col])]
raw_categorical_cols = [col for col in base_features if col not in raw_numeric_cols]
original_driver_set = set(original["Driver"].astype(str)) if original is not None and "Driver" in original else set()

if CV_ROW_LIMIT is not None and len(train) > CV_ROW_LIMIT:
    cv_train = (
        train.groupby(TARGET, group_keys=False)
        .apply(lambda x: x.sample(min(len(x), max(1, int(CV_ROW_LIMIT * len(x) / len(train)))), random_state=RANDOM_STATE))
        .sample(frac=1.0, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )
else:
    cv_train = train.copy()

cards = [
    metric_card("Train rows", f"{len(train):,}", f"CV rows: {len(cv_train):,}", COLORS["green"]),
    metric_card("Test rows", f"{len(test):,}", "submission target rows", COLORS["blue"]),
    metric_card("Train positive rate", f"{train[TARGET].mean():.2%}", TARGET, COLORS["red"]),
    metric_card("Driver cardinality", f"{train['Driver'].nunique():,}", f"test: {test['Driver'].nunique():,}", COLORS["gold"]),
]
if original is not None:
    cards.append(metric_card("Original rows", f"{len(original):,}", f"positive: {original[TARGET].mean():.2%}", COLORS["muted"]))

display_cards(cards, columns=min(5, len(cards)))

driver_inventory = pd.DataFrame({
    "Dataset": ["Train", "Test", "Original" if original is not None else "Original missing"],
    "Rows": [len(train), len(test), len(original) if original is not None else 0],
    "Unique_Drivers": [train["Driver"].nunique(), test["Driver"].nunique(), original["Driver"].nunique() if original is not None else 0],
    "D_Code_Rate": [
        train["Driver"].astype(str).str.match(r"^D\d+$").mean(),
        test["Driver"].astype(str).str.match(r"^D\d+$").mean(),
        original["Driver"].astype(str).str.match(r"^D\d+$").mean() if original is not None else np.nan,
    ],
    "Three_Letter_Rate": [
        train["Driver"].astype(str).str.match(r"^[A-Z]{3}$").mean(),
        test["Driver"].astype(str).str.match(r"^[A-Z]{3}$").mean(),
        original["Driver"].astype(str).str.match(r"^[A-Z]{3}$").mean() if original is not None else np.nan,
    ],
})
display_table(driver_inventory, "Driver inventory", percent_cols=["D_Code_Rate", "Three_Letter_Rate"])

driver_support = train["Driver"].value_counts().rename_axis("Driver").reset_index(name="Rows")
driver_support["Row_Share"] = driver_support["Rows"] / len(train)
support_summary = pd.DataFrame({
    "Threshold": [">= 10 rows", ">= 50 rows", ">= 100 rows", ">= 250 rows", ">= 500 rows"],
    "Drivers": [(driver_support["Rows"] >= t).sum() for t in [10, 50, 100, 250, 500]],
    "Covered_Rows": [driver_support.loc[driver_support["Rows"] >= t, "Rows"].sum() for t in [10, 50, 100, 250, 500]],
})
support_summary["Covered_Row_Rate"] = support_summary["Covered_Rows"] / len(train)
display_table(support_summary, "Driver support coverage", percent_cols=["Covered_Row_Rate"])

## 3. Feature Block Definitions

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-left:6px solid #3D6F8A; border-radius:14px; padding:13px 15px; margin:10px 0 16px 0; color:#233127; line-height:1.58; box-shadow:0 8px 18px rgba(35,49,39,0.04);">Feature blocks are ordered by leakage risk. Physical race-state and string/count features are target-free; target encodings appear later and are evaluated only through fold-safe construction.</div>


### 3.1 Feature Engineering Blueprint

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-radius:16px; padding:16px 18px; margin:10px 0 18px 0; color:#233127; box-shadow:0 8px 18px rgba(35,49,39,0.04);">
  <div style="font-size:1.18rem; font-weight:900; margin-bottom:8px; color:#233127;">What the feature engineering is trying to express</div>
  <div style="font-size:14px; line-height:1.6; color:#4C5A4C; margin-bottom:14px;">The raw table describes a car at one lap. The engineered table tries to express five modeling ideas: where the car is in the race, how old the tyre is relative to that race state, whether the compound changes tyre-life meaning, whether the current lap is inside a plausible pit window, and whether Driver carries repeated strategic context.</div>
  <table style="width:100%; border-collapse:collapse; font-size:13.5px; line-height:1.48;">
    <thead>
      <tr style="background:#233127; color:#FFFCF8; text-align:left;">
        <th style="padding:10px 11px; border:1px solid #D9D0BF; width:18%;">Block</th>
        <th style="padding:10px 11px; border:1px solid #D9D0BF; width:32%;">What is created</th>
        <th style="padding:10px 11px; border:1px solid #D9D0BF; width:32%;">Why it should help</th>
        <th style="padding:10px 11px; border:1px solid #D9D0BF; width:18%;">Leakage rule</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#2F6B4F;">Race/tyre algebra</td>
        <td style="padding:10px 11px; border:1px solid #E2D8C7;">Estimated race length, remaining laps, remaining race progress, tyre-life ratios.</td>
        <td style="padding:10px 11px; border:1px solid #E2D8C7;">Pit risk is about relative race state, not only absolute lap number. TyreLife=20 means different things early vs late.</td>
        <td style="padding:10px 11px; border:1px solid #E2D8C7;">Target-free</td>
      </tr>
      <tr style="background:#F7F3EA;">
        <td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#7A6045;">Compound pressure</td>
        <td style="padding:10px 11px; border:1px solid #E2D8C7;">Compound hardness, dry/wet-like flags, soft/hard flags, tyre-life times hardness.</td>
        <td style="padding:10px 11px; border:1px solid #E2D8C7;">The same tyre age is not equally risky on SOFT, MEDIUM, HARD, INTERMEDIATE, and WET tyres.</td>
        <td style="padding:10px 11px; border:1px solid #E2D8C7;">Target-free</td>
      </tr>
      <tr>
        <td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#B5473C;">Nonlinear race state</td>
        <td style="padding:10px 11px; border:1px solid #E2D8C7;">TyreLife x RaceProgress, TyreLife x Stint, RaceProgress x Stint, degradation x tyre life, absolute deltas.</td>
        <td style="padding:10px 11px; border:1px solid #E2D8C7;">Pit decisions are conditional windows. Risk can jump when tyre age, stint, phase, and degradation align.</td>
        <td style="padding:10px 11px; border:1px solid #E2D8C7;">Target-free</td>
      </tr>
      <tr style="background:#F7F3EA;">
        <td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#3D6F8A;">Driver representation</td>
        <td style="padding:10px 11px; border:1px solid #E2D8C7;">Raw one-hot identity, D-code/string structure, original-vocabulary flag, frequency, Driver target prior, and split Driver/strategy interaction priors.</td>
        <td style="padding:10px 11px; border:1px solid #E2D8C7;">Driver may encode repeated strategy context or synthetic-generation priors, but each representation must earn its place through CV.</td>
        <td style="padding:10px 11px; border:1px solid #E2D8C7;">Target-free first; inner-OOF for each TE candidate</td>
      </tr>
    </tbody>
  </table>
</div>


In [ ]:
COMPOUND_HARDNESS = {
    "SOFT": 1.0,
    "MEDIUM": 2.0,
    "HARD": 3.0,
    "INTERMEDIATE": 4.0,
    "WET": 5.0,
}
COMPOUND_EXPECTED_LIFE = {
    "SOFT": 25.0,
    "MEDIUM": 35.0,
    "HARD": 45.0,
    "INTERMEDIATE": 30.0,
    "WET": 40.0,
}
COMPOUND_WINDOW_START = {
    "SOFT": 0.64,
    "MEDIUM": 0.68,
    "HARD": 0.72,
    "INTERMEDIATE": 0.62,
    "WET": 0.60,
}
COMPOUND_FAST_HARDNESS = {
    "SOFT": 0.0,
    "MEDIUM": 1.0,
    "HARD": 2.0,
    "INTERMEDIATE": 1.5,
    "WET": 2.5,
}


def _safe_divide(a, b):
    return np.divide(a, np.where(np.abs(b) < 1e-9, np.nan, b))


def add_base_interactions(df):
    out = df.copy()
    out["Driver"] = out["Driver"].astype(str).fillna("MISSING")
    out["Compound"] = out["Compound"].astype(str).fillna("MISSING")
    out["Race"] = out["Race"].astype(str).fillna("MISSING")
    out["Stint_Str"] = out["Stint"].astype("Int64").astype(str).replace("<NA>", "MISSING")
    out["LapNumber_Str"] = out["LapNumber"].astype("Int64").astype(str).replace("<NA>", "MISSING")
    out["Race_Compound"] = out["Race"] + "__" + out["Compound"]
    out["Driver_Compound"] = out["Driver"] + "__" + out["Compound"]
    out["Race_Stint"] = out["Race"] + "__" + out["Stint_Str"]
    out["Compound_Stint"] = out["Compound"] + "__" + out["Stint_Str"]
    out["Race_Lap"] = out["Race"] + "__" + out["LapNumber_Str"]
    return out


def add_race_tyre_algebra(df):
    out = df.copy()
    estimated_laps = _safe_divide(out["LapNumber"].astype(float), out["RaceProgress"].astype(float)).replace([np.inf, -np.inf], np.nan)
    out["EstimatedRaceLaps"] = estimated_laps.clip(lower=1, upper=120)
    out["LapsRemainingEstimate"] = (out["EstimatedRaceLaps"] - out["LapNumber"]).clip(lower=-10, upper=120)
    out["RemainingRaceProgress"] = (1.0 - out["RaceProgress"].astype(float)).clip(-0.1, 1.1)
    out["TyreLifeToLapNumber"] = _safe_divide(out["TyreLife"].astype(float), out["LapNumber"].astype(float))
    out["TyreLifeToRaceProgress"] = _safe_divide(out["TyreLife"].astype(float), out["RaceProgress"].astype(float))
    out["NormalizedTyreLifeEstimate"] = _safe_divide(out["TyreLife"].astype(float), out["EstimatedRaceLaps"].astype(float))
    return out


def add_compound_structure(df):
    out = df.copy()
    hardness = out["Compound"].map(COMPOUND_HARDNESS).fillna(2.5)
    out["CompoundHardness"] = hardness
    out["CompoundIsDry"] = out["Compound"].isin(["SOFT", "MEDIUM", "HARD"]).astype(int)
    out["CompoundIsWetLike"] = out["Compound"].isin(["INTERMEDIATE", "WET"]).astype(int)
    out["CompoundIsSoft"] = (out["Compound"] == "SOFT").astype(int)
    out["CompoundIsHard"] = (out["Compound"] == "HARD").astype(int)
    out["TyreLife_x_CompoundHardness"] = out["TyreLife"].astype(float) * hardness
    if "NormalizedTyreLifeEstimate" in out:
        out["NormalizedTyreLife_x_CompoundHardness"] = out["NormalizedTyreLifeEstimate"].astype(float) * hardness
    return out



def add_public_domain_features(df):
    out = df.copy()
    compound = out["Compound"].astype(str)
    expected_life = compound.map(COMPOUND_EXPECTED_LIFE).fillna(35.0).astype(float)
    window_start = compound.map(COMPOUND_WINDOW_START).fillna(0.68).astype(float)
    fast_hardness = compound.map(COMPOUND_FAST_HARDNESS).fillna(1.5).astype(float)

    tyre_life = out["TyreLife"].astype(float)
    lap_number = out["LapNumber"].astype(float)
    race_progress = out["RaceProgress"].astype(float).clip(0, 1.05)
    lap_delta = out["LapTime_Delta"].astype(float)
    cum_deg = out["Cumulative_Degradation"].astype(float)
    recent_deg = out.get("Recent_Degradation", pd.Series(0.0, index=out.index)).astype(float)
    position_change = out["Position_Change"].astype(float)

    out["ExpectedMaxLife"] = expected_life
    out["PublicCompoundHardness"] = fast_hardness
    out["TyreLifePct"] = _safe_divide(tyre_life, expected_life).clip(0, 3)
    out["PitWindowStartPct"] = window_start
    out["InCompoundPitWindow"] = (out["TyreLifePct"] >= window_start).astype(int)
    out["WindowOvershootPct"] = (out["TyreLifePct"] - window_start).clip(lower=0, upper=2)
    out["DegPerLap"] = _safe_divide(cum_deg, tyre_life + 1.0)
    out["RecentDegPerLap"] = _safe_divide(recent_deg, tyre_life + 1.0)
    out["AbsCumDeg"] = cum_deg.abs()
    out["AbsRecentDeg"] = recent_deg.abs()
    out["IsDegrading"] = (cum_deg < 0).astype(int)
    out["PositiveLapTimeDelta"] = lap_delta.clip(lower=0)
    out["NegativeLapTimeDelta"] = (-lap_delta).clip(lower=0)
    out["AgeXDelta"] = tyre_life * out["PositiveLapTimeDelta"]
    out["PositionLossXAge"] = position_change.clip(upper=0).abs() * tyre_life
    out["StintCapped"] = out["Stint"].astype(float).clip(1, 4)
    out["StintPhase"] = pd.cut(race_progress, bins=[-0.01, 0.25, 0.65, 1.05], labels=[0, 1, 2]).astype(float)
    out["LapNumberSq"] = (lap_number ** 2).clip(0, 10_000)
    out["TyreLifeSq"] = (tyre_life ** 2).clip(0, 10_000)
    out["StopUrgency"] = out["TyreLifePct"] * race_progress * (1.0 + 0.20 * out["StintCapped"])
    out["WindowPressure"] = out["InCompoundPitWindow"] * out["StintCapped"]
    out["TyreLifeMinusExpectedWindow"] = tyre_life - expected_life * window_start
    if "LapsRemainingEstimate" in out:
        out["LapsRemainingPct"] = _safe_divide(out["LapsRemainingEstimate"].astype(float), out["EstimatedRaceLaps"].astype(float)).clip(-0.5, 1.5)
        out["LateRaceOldTyre"] = out["TyreLifePct"] * (1.0 - out["LapsRemainingPct"].clip(0, 1))
    return out


def add_driver_structure(df):
    out = df.copy()
    driver = out["Driver"].astype(str)
    extracted = driver.str.extract(r"^D(\d+)$", expand=False)
    out["Driver_is_D_code"] = driver.str.match(r"^D\d+$").astype(int)
    out["Driver_is_3letter"] = driver.str.match(r"^[A-Z]{3}$").astype(int)
    out["Driver_num"] = pd.to_numeric(extracted, errors="coerce").fillna(-1)
    out["Driver_num_missing"] = (out["Driver_num"] < 0).astype(int)
    bins = [-2, -0.5, 99, 199, 399, 699, 9999]
    out["Driver_num_bin"] = pd.cut(out["Driver_num"], bins=bins, labels=False, include_lowest=True).astype(float)
    return out


def add_driver_original_vocab(df, original_drivers=None):
    original_drivers = original_drivers or set()
    out = df.copy()
    out["Driver_in_original"] = out["Driver"].astype(str).isin(original_drivers).astype(int)
    return out


def add_nonlinear_interactions(df):
    out = df.copy()
    out["TyreLife_x_RaceProgress"] = out["TyreLife"].astype(float) * out["RaceProgress"].astype(float)
    out["TyreLife_x_Stint"] = out["TyreLife"].astype(float) * out["Stint"].astype(float)
    out["RaceProgress_x_Stint"] = out["RaceProgress"].astype(float) * out["Stint"].astype(float)
    out["CumulativeDegradation_x_TyreLife"] = out["Cumulative_Degradation"].astype(float) * out["TyreLife"].astype(float)
    out["LapTimeDelta_x_TyreLife"] = out["LapTime_Delta"].astype(float) * out["TyreLife"].astype(float)
    out["AbsPositionChange"] = out["Position_Change"].abs()
    out["PositionLossFlag"] = (out["Position_Change"] > 0).astype(int)
    out["PositionGainFlag"] = (out["Position_Change"] < 0).astype(int)
    out["AbsLapTimeDelta"] = out["LapTime_Delta"].abs()
    return out


def fit_count_maps(fit_df, columns, weight=None):
    w = np.ones(len(fit_df), dtype=float) if weight is None else np.asarray(weight, dtype=float)
    maps = {}
    for col in columns:
        tmp = pd.DataFrame({col: fit_df[col].astype(str), "__w": w})
        maps[col] = tmp.groupby(col, observed=True)["__w"].sum()
    return maps


def apply_count_maps(df, count_maps):
    out = df.copy()
    for col, counts in count_maps.items():
        mapped = out[col].astype(str).map(counts).fillna(0).astype(float)
        out[f"{col}_count"] = mapped
        out[f"{col}_freq_log1p"] = np.log1p(mapped)
        if col == "Driver":
            out["Driver_is_rare_50"] = (mapped < 50).astype(int)
    return out


def _weighted_mean_std(values, weight):
    x = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)
    w = np.ones(len(x), dtype=float) if weight is None else np.asarray(weight, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if not mask.any():
        return {"mean": 0.0, "std": 1.0}
    x = x[mask]
    w = w[mask]
    mean = np.average(x, weights=w)
    var = np.average((x - mean) ** 2, weights=w)
    std = float(np.sqrt(max(var, 0.0)))
    return {"mean": float(mean), "std": std if std > 0 else 1.0}


def fit_global_stats(fit_df, value_cols, weight=None):
    return {col: _weighted_mean_std(fit_df[col], weight) for col in value_cols}


def fit_group_stats(fit_df, specs, weight=None):
    w = np.ones(len(fit_df), dtype=float) if weight is None else np.asarray(weight, dtype=float)
    stats = {}
    for group_col, value_cols in specs.items():
        stats[group_col] = {}
        for value_col in value_cols:
            work = pd.DataFrame({
                group_col: fit_df[group_col].astype(str),
                "__x": pd.to_numeric(fit_df[value_col], errors="coerce"),
                "__w": w,
            }).dropna(subset=["__x"])
            work = work[np.isfinite(work["__x"]) & np.isfinite(work["__w"]) & (work["__w"] > 0)]
            work["__wx"] = work["__w"] * work["__x"]
            work["__wx2"] = work["__w"] * work["__x"] * work["__x"]
            grouped = work.groupby(group_col, observed=True).agg(
                weight_sum=("__w", "sum"),
                wx=("__wx", "sum"),
                wx2=("__wx2", "sum"),
            )
            mean = grouped["wx"] / grouped["weight_sum"]
            var = (grouped["wx2"] / grouped["weight_sum"] - mean * mean).clip(lower=0)
            frame = pd.DataFrame({"mean": mean, "std": np.sqrt(var).replace(0, np.nan)})
            stats[group_col][value_col] = frame
    return stats


def apply_group_stats(df, group_stats, global_stats):
    out = df.copy()
    for group_col, value_map in group_stats.items():
        for value_col, stats in value_map.items():
            mean_map = stats["mean"]
            std_map = stats["std"]
            mean = out[group_col].map(mean_map).fillna(global_stats[value_col]["mean"])
            std = out[group_col].map(std_map).fillna(global_stats[value_col]["std"] or 1.0)
            out[f"{value_col}_z_by_{group_col}"] = (out[value_col].astype(float) - mean) / (std + 1e-6)
    return out


def fit_target_maps(fit_df, columns, target, weight=None, alpha=80):
    y = fit_df[target].astype(float).to_numpy()
    w = np.ones(len(fit_df), dtype=float) if weight is None else np.asarray(weight, dtype=float)
    prior = np.average(y, weights=w)
    maps = {}
    tmp = fit_df.copy()
    tmp["__y"] = y
    tmp["__w"] = w
    tmp["__yw"] = y * w
    for col in columns:
        grouped = tmp.groupby(col, observed=True).agg(weight_sum=("__w", "sum"), target_sum=("__yw", "sum"))
        maps[col] = ((grouped["target_sum"] + alpha * prior) / (grouped["weight_sum"] + alpha)).to_dict()
    return maps, prior


def apply_target_maps(df, target_maps, prior, suffix="te"):
    out = df.copy()
    for col, mapping in target_maps.items():
        out[f"{col}_{suffix}"] = out[col].map(mapping).fillna(prior).astype(float)
    return out


def add_inner_oof_target_encoding(
    fit_df,
    target,
    columns,
    sample_weight=None,
    alpha=80,
    n_splits=5,
    seed=RANDOM_STATE,
    suffix="te",
):
    out = fit_df.copy()
    y = out[target].astype(int).to_numpy()
    w = np.ones(len(out), dtype=float) if sample_weight is None else np.asarray(sample_weight, dtype=float)
    class_counts = np.bincount(y)
    min_class = class_counts[class_counts > 0].min() if len(class_counts) else 0
    inner_splits = int(min(n_splits, min_class))
    if inner_splits < 2:
        maps, prior = fit_target_maps(out, columns, target, w, alpha=alpha)
        return apply_target_maps(out, maps, prior, suffix=suffix)

    skf = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=seed)
    for col in columns:
        encoded = np.zeros(len(out), dtype=float)
        for tr_idx, va_idx in skf.split(out, y):
            maps, prior = fit_target_maps(
                out.iloc[tr_idx],
                [col],
                target,
                w[tr_idx],
                alpha=alpha,
            )
            encoded[va_idx] = out.iloc[va_idx][col].map(maps[col]).fillna(prior).astype(float)
        out[f"{col}_{suffix}"] = encoded
    return out

BLOCK_NUMERIC_FEATURES = {
    "race_tyre_algebra": [
        "EstimatedRaceLaps", "LapsRemainingEstimate", "RemainingRaceProgress",
        "TyreLifeToLapNumber", "TyreLifeToRaceProgress", "NormalizedTyreLifeEstimate",
    ],
    "compound_structure": [
        "CompoundHardness", "CompoundIsDry", "CompoundIsWetLike", "CompoundIsSoft",
        "CompoundIsHard", "TyreLife_x_CompoundHardness", "NormalizedTyreLife_x_CompoundHardness",
    ],
    "public_domain_features": [
        "ExpectedMaxLife", "PublicCompoundHardness", "TyreLifePct", "PitWindowStartPct",
        "InCompoundPitWindow", "WindowOvershootPct", "DegPerLap", "RecentDegPerLap",
        "AbsCumDeg", "AbsRecentDeg", "IsDegrading", "PositiveLapTimeDelta",
        "NegativeLapTimeDelta", "AgeXDelta", "PositionLossXAge", "StintCapped",
        "StintPhase", "LapNumberSq", "TyreLifeSq", "StopUrgency", "WindowPressure",
        "TyreLifeMinusExpectedWindow", "LapsRemainingPct", "LateRaceOldTyre",
    ],
    "driver_structure": [
        "Driver_is_D_code", "Driver_is_3letter", "Driver_num", "Driver_num_bin",
        "Driver_num_missing",
    ],
    "driver_original_vocab": ["Driver_in_original"],
    "frequency_encoding": [
        "Driver_count", "Driver_freq_log1p", "Driver_is_rare_50", "Race_count", "Race_freq_log1p",
        "Driver_Compound_count", "Driver_Compound_freq_log1p", "Race_Compound_count",
        "Race_Compound_freq_log1p", "Race_Stint_count", "Race_Stint_freq_log1p",
        "Compound_Stint_count", "Compound_Stint_freq_log1p",
    ],
    "context_normalization": [
        "TyreLife_z_by_Race", "LapNumber_z_by_Race", "LapTime_Delta_z_by_Race",
        "Cumulative_Degradation_z_by_Race", "TyreLife_z_by_Race_Compound", "LapTime_Delta_z_by_Race_Compound",
        "TyreLife_z_by_Race_Lap", "Position_z_by_Race_Lap", "LapTime_Delta_z_by_Race_Lap",
        "TyreLife_z_by_Compound", "Cumulative_Degradation_z_by_Compound",
    ],
    "nonlinear_interactions": [
        "TyreLife_x_RaceProgress", "TyreLife_x_Stint", "RaceProgress_x_Stint",
        "CumulativeDegradation_x_TyreLife", "LapTimeDelta_x_TyreLife", "AbsPositionChange",
        "PositionLossFlag", "PositionGainFlag", "AbsLapTimeDelta",
    ],
    "oof_target_encoding": ["Driver_te", "Race_te"],
    "driver_compound_te": ["Driver_Compound_te"],
    "race_compound_te": ["Race_Compound_te"],
    "race_stint_te": ["Race_Stint_te"],
    "compound_stint_te": ["Compound_Stint_te"],
    "interaction_target_encoding": [
        "Driver_Compound_te", "Race_Compound_te", "Race_Stint_te", "Compound_Stint_te",
    ],
}

FREQUENCY_COLUMNS = ["Driver", "Race", "Driver_Compound", "Race_Compound", "Race_Stint", "Compound_Stint"]
CONTEXT_GROUP_SPECS = {
    "Race": ["TyreLife", "LapNumber", "LapTime_Delta", "Cumulative_Degradation"],
    "Race_Compound": ["TyreLife", "LapTime_Delta"],
    "Race_Lap": ["TyreLife", "Position", "LapTime_Delta"],
    "Compound": ["TyreLife", "Cumulative_Degradation"],
}
TARGET_ENCODING_COLUMNS = ["Driver", "Race"]
INTERACTION_TARGET_COLUMNS = ["Driver_Compound", "Race_Compound", "Race_Stint", "Compound_Stint"]
INTERACTION_TARGET_BLOCK_COLUMNS = {
    "driver_compound_te": ["Driver_Compound"],
    "race_compound_te": ["Race_Compound"],
    "race_stint_te": ["Race_Stint"],
    "compound_stint_te": ["Compound_Stint"],
}

## 4. Fold-Safe Preprocessing and CV Utilities

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-left:6px solid #B5473C; border-radius:14px; padding:13px 15px; margin:10px 0 16px 0; color:#233127; line-height:1.58; box-shadow:0 8px 18px rgba(35,49,39,0.04);">The validation contract is strict: each fold fits count maps, group statistics, and target encodings only on the training side. Validation rows are transformed from those fitted objects, so OOF scores cannot see their own labels.</div>


In [ ]:
@dataclass
class ExperimentConfig:
    name: str
    label: str
    blocks: list[str]
    include_driver_ohe: bool = True
    model_kind: str = "lgbm"
    original_weight: float = 0.0


def prepare_static_features(df, blocks):
    out = add_base_interactions(df)
    if "race_tyre_algebra" in blocks:
        out = add_race_tyre_algebra(out)
    if "compound_structure" in blocks:
        out = add_compound_structure(out)
    if "public_domain_features" in blocks:
        out = add_public_domain_features(out)
    if "driver_structure" in blocks:
        out = add_driver_structure(out)
    if "driver_original_vocab" in blocks:
        out = add_driver_original_vocab(out, original_driver_set)
    if "nonlinear_interactions" in blocks:
        out = add_nonlinear_interactions(out)
    return out


def fit_fold_features(fit_df, valid_df, test_df, blocks, target, fit_weight=None):
    fit_x = prepare_static_features(fit_df, blocks)
    valid_x = prepare_static_features(valid_df, blocks)
    test_x = prepare_static_features(test_df, blocks)

    if "frequency_encoding" in blocks:
        count_maps = fit_count_maps(fit_x, FREQUENCY_COLUMNS, weight=fit_weight)
        fit_x = apply_count_maps(fit_x, count_maps)
        valid_x = apply_count_maps(valid_x, count_maps)
        test_x = apply_count_maps(test_x, count_maps)

    if "context_normalization" in blocks:
        value_cols = sorted({value for values in CONTEXT_GROUP_SPECS.values() for value in values})
        global_stats = fit_global_stats(fit_x, value_cols, weight=fit_weight)
        group_stats = fit_group_stats(fit_x, CONTEXT_GROUP_SPECS, weight=fit_weight)
        fit_x = apply_group_stats(fit_x, group_stats, global_stats)
        valid_x = apply_group_stats(valid_x, group_stats, global_stats)
        test_x = apply_group_stats(test_x, group_stats, global_stats)

    if "oof_target_encoding" in blocks:
        fit_x = add_inner_oof_target_encoding(
            fit_x,
            target=target,
            columns=TARGET_ENCODING_COLUMNS,
            sample_weight=fit_weight,
            alpha=80,
            n_splits=INNER_TE_SPLITS,
            suffix="te",
        )
        maps, prior = fit_target_maps(fit_x, TARGET_ENCODING_COLUMNS, target, fit_weight, alpha=80)
        valid_x = apply_target_maps(valid_x, maps, prior, suffix="te")
        test_x = apply_target_maps(test_x, maps, prior, suffix="te")

    interaction_columns = []
    if "interaction_target_encoding" in blocks:
        interaction_columns.extend(INTERACTION_TARGET_COLUMNS)
    for block_name, block_columns in INTERACTION_TARGET_BLOCK_COLUMNS.items():
        if block_name in blocks:
            interaction_columns.extend(block_columns)
    interaction_columns = list(dict.fromkeys(interaction_columns))

    if interaction_columns:
        fit_x = add_inner_oof_target_encoding(
            fit_x,
            target=target,
            columns=interaction_columns,
            sample_weight=fit_weight,
            alpha=320,
            n_splits=INNER_TE_SPLITS,
            suffix="te",
        )
        maps, prior = fit_target_maps(fit_x, interaction_columns, target, fit_weight, alpha=320)
        valid_x = apply_target_maps(valid_x, maps, prior, suffix="te")
        test_x = apply_target_maps(test_x, maps, prior, suffix="te")

    return fit_x, valid_x, test_x


def select_numeric_features(blocks, available_columns=None):
    features = list(raw_numeric_cols)
    for block in blocks:
        features.extend(BLOCK_NUMERIC_FEATURES.get(block, []))
    features = list(dict.fromkeys(features))
    if available_columns is not None:
        available = set(available_columns)
        features = [col for col in features if col in available]
    return features


def select_categorical_features(include_driver_ohe=True):
    cats = ["Compound", "Race"]
    if include_driver_ohe:
        cats = ["Driver"] + cats
    return cats


def build_preprocessor(numeric_features, categorical_features, scale_numeric=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler(with_mean=False)))
    numeric_pipe = Pipeline(numeric_steps)
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
        ("onehot", make_ohe()),
    ])
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_features),
            ("cat", categorical_pipe, categorical_features),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


def make_model(kind):
    if kind == "logreg":
        return LogisticRegression(
            C=1.5,
            max_iter=800,
            solver="saga",
            n_jobs=-1,
            class_weight=None,
            random_state=RANDOM_STATE,
        )
    if kind == "lgbm":
        if lgb is None:
            raise ImportError("LightGBM is not available")
        return lgb.LGBMClassifier(
            objective="binary",
            metric="auc",
            n_estimators=LGBM_N_ESTIMATORS,
            learning_rate=0.06 if FAST_MODE else 0.03,
            num_leaves=48 if FAST_MODE else 127,
            max_depth=-1,
            min_child_samples=80,
            subsample=0.88 if FAST_MODE else 0.80,
            colsample_bytree=0.88 if FAST_MODE else 0.80,
            reg_alpha=0.03,
            reg_lambda=0.35,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
        )
    if kind == "xgb":
        if xgb is None:
            raise ImportError("XGBoost is not available")
        return xgb.XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            device="cuda" if USE_GPU else "cpu",
            n_estimators=XGB_N_ESTIMATORS,
            learning_rate=0.06 if FAST_MODE else 0.03,
            max_depth=5 if FAST_MODE else 7,
            min_child_weight=30 if FAST_MODE else 5,
            subsample=0.88 if FAST_MODE else 0.80,
            colsample_bytree=0.88 if FAST_MODE else 0.80,
            reg_alpha=0.02,
            reg_lambda=1.2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    if kind == "cat":
        if CatBoostClassifier is None:
            raise ImportError("CatBoost is not available")
        return CatBoostClassifier(
            iterations=CAT_N_ITERATIONS,
            learning_rate=0.06 if FAST_MODE else 0.05,
            depth=6 if FAST_MODE else 7,
            loss_function="Logloss",
            eval_metric="AUC",
            task_type="GPU" if USE_GPU else "CPU",
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )
    raise ValueError(f"Unknown model kind: {kind}")


def fit_predict_model(kind, model, X_fit, y_fit, X_valid, y_valid, X_test, sample_weight=None):
    if kind == "lgbm":
        callbacks = [lgb.early_stopping(80 if not FAST_MODE else 25, verbose=False)]
        model.fit(
            X_fit, y_fit,
            sample_weight=sample_weight,
            eval_set=[(X_valid, y_valid)],
            callbacks=callbacks,
        )
    elif kind == "xgb":
        try:
            model.fit(
                X_fit, y_fit,
                sample_weight=sample_weight,
                eval_set=[(X_valid, y_valid)],
                early_stopping_rounds=25 if FAST_MODE else 100,
                verbose=False,
            )
        except TypeError:
            model.fit(X_fit, y_fit, sample_weight=sample_weight, eval_set=[(X_valid, y_valid)], verbose=False)
    elif kind == "cat":
        model.fit(X_fit, y_fit, sample_weight=sample_weight, eval_set=(X_valid, y_valid), verbose=False)
    else:
        model.fit(X_fit, y_fit, sample_weight=sample_weight)
    valid_pred = model.predict_proba(X_valid)[:, 1]
    test_pred = model.predict_proba(X_test)[:, 1]
    return valid_pred, test_pred, model


def make_original_frame(feature_columns, target, weight):
    if original is None or weight <= 0:
        return None, None
    missing_cols = [col for col in feature_columns + [target] if col not in original.columns]
    if missing_cols:
        return None, None
    frame = original[feature_columns + [target]].copy()
    sample_weight = np.full(len(frame), weight, dtype=float)
    return frame, sample_weight


def run_oof_experiment(config, train_frame=None, test_frame=None, verbose=True):
    train_frame = cv_train if train_frame is None else train_frame
    test_frame = test if test_frame is None else test_frame
    y = train_frame[TARGET].astype(int).to_numpy()
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    numeric_features = None
    categorical_features = select_categorical_features(config.include_driver_ohe)
    feature_columns = list(dict.fromkeys(base_features))
    oof = np.zeros(len(train_frame), dtype=float)
    test_pred = np.zeros(len(test_frame), dtype=float)
    fold_rows = []
    fitted_models = []

    original_frame, original_weight = make_original_frame(feature_columns, TARGET, config.original_weight)

    for fold, (fit_idx, valid_idx) in enumerate(skf.split(train_frame, y), start=1):
        fold_fit = train_frame.iloc[fit_idx].copy()
        fold_valid = train_frame.iloc[valid_idx].copy()
        fold_weight = np.ones(len(fold_fit), dtype=float)

        if original_frame is not None:
            fold_fit = pd.concat([fold_fit, original_frame], ignore_index=True)
            fold_weight = np.concatenate([fold_weight, original_weight])

        fit_x, valid_x, test_x = fit_fold_features(fold_fit, fold_valid, test_frame, config.blocks, TARGET, fold_weight)
        numeric_features = select_numeric_features(config.blocks, available_columns=fit_x.columns)
        preprocessor = build_preprocessor(
            numeric_features=numeric_features,
            categorical_features=categorical_features,
            scale_numeric=(config.model_kind == "logreg"),
        )
        X_fit = preprocessor.fit_transform(fit_x)
        X_valid = preprocessor.transform(valid_x)
        X_test = preprocessor.transform(test_x)
        y_fit = fold_fit[TARGET].astype(int).to_numpy()
        y_valid = fold_valid[TARGET].astype(int).to_numpy()

        model = make_model(config.model_kind)
        valid_pred, fold_test_pred, fitted_model = fit_predict_model(
            config.model_kind, model, X_fit, y_fit, X_valid, y_valid, X_test, sample_weight=fold_weight
        )
        oof[valid_idx] = valid_pred
        test_pred += fold_test_pred / N_SPLITS
        fold_auc = roc_auc_score(y_valid, valid_pred)
        fold_logloss = log_loss(y_valid, np.clip(valid_pred, 1e-6, 1 - 1e-6))
        fold_rows.append({"Experiment": config.name, "Fold": fold, "AUC": fold_auc, "LogLoss": fold_logloss})
        fitted_models.append((preprocessor, fitted_model, numeric_features, categorical_features))
        if verbose:
            print(f"{config.name} | fold {fold}/{N_SPLITS} | AUC {fold_auc:.5f}")

    result = {
        "config": config,
        "oof": oof,
        "test_pred": test_pred,
        "folds": pd.DataFrame(fold_rows),
        "auc": roc_auc_score(y, oof),
        "logloss": log_loss(y, np.clip(oof, 1e-6, 1 - 1e-6)),
        "models": fitted_models,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
    }
    return result


def summarize_results(results, baseline_auc=None):
    rows = []
    previous_auc = None
    for result in results:
        cfg = result["config"]
        auc = result["auc"]
        delta_previous = np.nan if previous_auc is None else auc - previous_auc
        delta_baseline = np.nan if baseline_auc is None else auc - baseline_auc
        rows.append({
            "Experiment": cfg.name,
            "Added_Block": cfg.label,
            "Model": cfg.model_kind,
            "Driver_OHE": cfg.include_driver_ohe,
            "Original_Weight": cfg.original_weight,
            "OOF_AUC": auc,
            "Delta_vs_Previous": delta_previous,
            "Delta_vs_Baseline": delta_baseline,
            "OOF_LogLoss": result["logloss"],
        })
        previous_auc = auc
    return pd.DataFrame(rows)


def plot_ladder(summary, title):
    plot_df = summary.copy()
    plt.figure(figsize=(11, max(4, 0.45 * len(plot_df))))
    ax = sns.barplot(data=plot_df, y="Experiment", x="OOF_AUC", color=COLORS["green"])
    ax.set_title(title, fontsize=16, fontweight="bold", color=COLORS["ink"])
    ax.set_xlabel("OOF ROC-AUC")
    ax.set_ylabel("")
    xmin = max(0.5, plot_df["OOF_AUC"].min() - 0.01)
    xmax = min(1.0, plot_df["OOF_AUC"].max() + 0.01)
    ax.set_xlim(xmin, xmax)
    for patch, value in zip(ax.patches, plot_df["OOF_AUC"]):
        ax.text(value + 0.0005, patch.get_y() + patch.get_height() / 2, f"{value:.5f}", va="center", fontsize=10, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 5. Transparent OHE Baseline

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-left:6px solid #2F6B4F; border-radius:14px; padding:13px 15px; margin:10px 0 16px 0; color:#233127; line-height:1.58; box-shadow:0 8px 18px rgba(35,49,39,0.04);">The first question is deliberately simple: does Driver add anything at all? A no-Driver logistic baseline is compared against a Driver one-hot baseline before the stronger ladder begins.</div>


In [ ]:
baseline_configs = [
    ExperimentConfig("B00_no_driver_logreg", "Raw OHE without Driver", [], include_driver_ohe=False, model_kind="logreg"),
    ExperimentConfig("B01_driver_ohe_logreg", "Add Driver one-hot", [], include_driver_ohe=True, model_kind="logreg"),
]

baseline_results = [run_oof_experiment(cfg, verbose=True) for cfg in baseline_configs]
baseline_summary = summarize_results(baseline_results, baseline_auc=baseline_results[0]["auc"])
display_table(baseline_summary, "Transparent logistic baselines")
plot_ladder(baseline_summary, "Does raw Driver OHE add signal?")

## 6. Driver-Centered CV Ladder

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-left:6px solid #C7772C; border-radius:14px; padding:13px 15px; margin:10px 0 16px 0; color:#233127; line-height:1.58; box-shadow:0 8px 18px rgba(35,49,39,0.04);">This is the core evidence section. The Driver story is split into two complementary ladders: a <b>raw identity ladder</b> that asks how much Driver one-hot helps, and a <b>compressed Driver ladder</b> that tests string structure, frequency, inner-OOF target encoding, and interaction target encoding without raw Driver one-hot. This separation makes the marginal value of engineered Driver signal easier to read.</div>


### 6.1 Driver Ladder: What Changes at Each Step

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-radius:16px; padding:16px 18px; margin:10px 0 18px 0; color:#233127; box-shadow:0 8px 18px rgba(35,49,39,0.04);">
  <div style="font-size:1.18rem; font-weight:900; margin-bottom:8px; color:#233127;">The controlled experiment</div>
  <div style="font-size:14px; line-height:1.6; color:#4C5A4C; margin-bottom:14px;">Every row uses the same fixed race-state foundation: race/tyre algebra, compound pressure, and nonlinear race-state interactions. Only the Driver representation changes, so each CV delta is read as Driver-feature evidence.</div>
  <table style="width:100%; border-collapse:collapse; font-size:13.5px; line-height:1.48;">
    <thead>
      <tr style="background:#233127; color:#FFFCF8; text-align:left;">
        <th style="padding:10px 11px; border:1px solid #D9D0BF; width:14%;">Step</th>
        <th style="padding:10px 11px; border:1px solid #D9D0BF; width:30%;">Exactly what is added</th>
        <th style="padding:10px 11px; border:1px solid #D9D0BF; width:36%;">Why this Driver signal might exist</th>
        <th style="padding:10px 11px; border:1px solid #D9D0BF; width:20%;">How to read CV</th>
      </tr>
    </thead>
    <tbody>
      <tr><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900;">A00 / B00</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">No Driver information. Race-state foundation only.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">This is the reference point: how far race/tyre/compound state can go without Driver.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Baseline</td></tr>
      <tr style="background:#F7F3EA;"><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#2F6B4F;">A01</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Raw Driver one-hot indicators.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Tests whether Driver identity itself carries a repeated strategy or generator prior.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Raw identity lift.</td></tr>
      <tr><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#3D6F8A;">B01</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">D-code flag, 3-letter flag, numeric Driver ID, ID bin, missing-number flag.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Separates synthetic-style D IDs from real-style 3-letter codes without raw one-hot identity.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">String-structure lift.</td></tr>
      <tr style="background:#F7F3EA;"><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#7A6045;">B02</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Driver-in-original-vocabulary flag.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">This uses the original dataset vocabulary, so it is separated from pure string structure.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">External-vocabulary lift.</td></tr>
      <tr><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#C7772C;">B03</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Driver count, log frequency, rare-driver flag, Race count, Driver-Compound count.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Support size matters for high-cardinality categories; rare contexts are noisier.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Support/rarity lift.</td></tr>
      <tr style="background:#F7F3EA;"><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#B5473C;">B04</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Inner-OOF smoothed target priors for Driver and Race.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Compresses many categories into a prior while preventing rows from encoding their own labels.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">TE must beat simpler blocks.</td></tr>
      <tr><td style="padding:10px 11px; border:1px solid #E2D8C7; font-weight:900; color:#3D6F8A;">B05-B08</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Driver-Compound, Race-Compound, Race-Stint, and Compound-Stint TE tested one at a time.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Strategy is conditional, but interaction support is smaller and variance can dominate.</td><td style="padding:10px 11px; border:1px solid #E2D8C7;">Accept only positive OOF lift.</td></tr>
    </tbody>
  </table>
</div>


In [ ]:
INTERACTION_BRANCH_EXPERIMENTS = {
    "B05_driver_compound_te",
    "B06_race_compound_te",
    "B07_race_stint_te",
    "B08_compound_stint_te",
}


def enrich_ladder_summary(summary, score_col="OOF_AUC"):
    out = summary.copy()
    base = out[score_col].iloc[0]
    out["Delta_vs_No_Driver"] = out[score_col] - base
    out["Cumulative_Best_AUC"] = out[score_col].cummax()
    out["Accepted_Best"] = out[score_col].eq(out["Cumulative_Best_AUC"])
    out["CV_Read"] = np.where(
        out["Delta_vs_Previous"].fillna(0) > 0,
        "✅ improves previous",
        np.where(out.index == 0, "baseline", "⚠️ does not improve previous"),
    )
    if "Experiment" in out.columns:
        branch_mask = out["Experiment"].isin(INTERACTION_BRANCH_EXPERIMENTS)
        out.loc[branch_mask, "CV_Read"] = "branch candidate; see Delta_vs_Parent"
        out.loc[branch_mask, "Accepted_Best"] = False
    return out


def plot_driver_ladder(summary, title):
    plot_df = summary.copy().reset_index(drop=True)
    x = np.arange(len(plot_df))
    fig, ax = plt.subplots(figsize=(12, 5.2))
    ax.plot(x, plot_df["OOF_AUC"], marker="o", linewidth=2.4, color=COLORS["blue"], label="Raw step AUC")
    ax.plot(x, plot_df["Cumulative_Best_AUC"], marker="s", linewidth=2.4, color=COLORS["green"], label="Cumulative best")
    for idx, row in plot_df.iterrows():
        color = COLORS["green"] if row["Accepted_Best"] else COLORS["muted"]
        ax.text(idx, row["OOF_AUC"] + 0.00035, f"{row['OOF_AUC']:.5f}", ha="center", fontsize=9, fontweight="bold", color=color)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["Experiment"], rotation=25, ha="right")
    ax.set_ylabel("OOF ROC-AUC")
    ax.set_title(title, fontsize=16, fontweight="bold", color=COLORS["ink"])
    ymin = max(0.5, plot_df["OOF_AUC"].min() - 0.004)
    ymax = min(1.0, plot_df["OOF_AUC"].max() + 0.004)
    ax.set_ylim(ymin, ymax)
    ax.legend(loc="lower right")
    ax.grid(axis="y", alpha=0.35)
    plt.tight_layout()
    plt.show()


def show_driver_takeaway(summary):
    best_row = summary.loc[summary["OOF_AUC"].idxmax()]
    base_auc = summary["OOF_AUC"].iloc[0]
    best_gain = best_row["OOF_AUC"] - base_auc
    cards = [
        metric_card("No-Driver foundation", f"{base_auc:.5f}", "fixed race-state features", COLORS["blue"]),
        metric_card("Best Driver step", best_row["Experiment"], best_row["Added_Block"], COLORS["green"]),
        metric_card("Best gain", f"{best_gain:+.5f}", "AUC vs no-Driver foundation", COLORS["gold"]),
        metric_card("Accepted steps", f"{int(summary['Accepted_Best'].sum())}/{len(summary)}", "cumulative-best checkpoints", COLORS["red"]),
    ]
    display_cards(cards, columns=4)

def read_small_delta(delta, tolerance=0.0):
    if delta > tolerance:
        return "small positive indication"
    if delta < -tolerance:
        return "rejected in this CV run"
    return "neutral within this run"


def show_driver_argument(identity_summary, compressed_summary):
    raw_lift = identity_summary["OOF_AUC"].iloc[1] - identity_summary["OOF_AUC"].iloc[0]
    compressed_best = compressed_summary.loc[compressed_summary["OOF_AUC"].idxmax()]
    compressed_lift = compressed_best["OOF_AUC"] - compressed_summary["OOF_AUC"].iloc[0]
    raw_read = read_small_delta(raw_lift)
    compressed_read = read_small_delta(compressed_lift)

    cards = [
        metric_card("Raw identity lift", f"{raw_lift:+.5f}", "A01 Driver OHE vs A00", COLORS["green"] if raw_lift > 0 else COLORS["red"]),
        metric_card("Compressed best lift", f"{compressed_lift:+.5f}", str(compressed_best["Experiment"]), COLORS["green"] if compressed_lift > 0 else COLORS["red"]),
        metric_card("TE guardrail", "inner OOF", "training rows do not encode from themselves", COLORS["blue"]),
        metric_card("Selection", "OOF best", "small deltas need confirmation", COLORS["gold"]),
    ]
    display_cards(cards, columns=4)

    argument_table = pd.DataFrame([
        {
            "Claim": "Raw Driver identity adds marginal signal",
            "Evidence": f"A01 - A00 = {raw_lift:+.5f} AUC",
            "CV_Read": raw_read,
        },
        {
            "Claim": "Compressed Driver pieces recover some signal without raw OHE",
            "Evidence": f"best compressed - B00 = {compressed_lift:+.5f} AUC",
            "CV_Read": compressed_read,
        },
        {
            "Claim": "Target encoding is evaluated conservatively",
            "Evidence": "target-encoded training rows use inner OOF values",
            "CV_Read": "leakage guardrail",
        },
        {
            "Claim": "Interaction TE is high-variance and must be split",
            "Evidence": "Driver-Compound, Race-Compound, Race-Stint, and Compound-Stint are tested separately",
            "CV_Read": "accept only OOF lift",
        },
    ])
    display_table(argument_table, "Driver argument summary")


def make_interaction_branch_table(summary, parent_name="B04_driver_te"):
    interaction_names = sorted(INTERACTION_BRANCH_EXPERIMENTS)
    if parent_name not in set(summary["Experiment"]):
        return pd.DataFrame()
    parent_auc = summary.loc[summary["Experiment"].eq(parent_name), "OOF_AUC"].iloc[0]
    branch = summary.loc[summary["Experiment"].isin(interaction_names)].copy()
    if branch.empty:
        return branch
    branch["Parent"] = parent_name
    branch["Delta_vs_Parent"] = branch["OOF_AUC"] - parent_auc
    branch["Decision"] = np.where(branch["Delta_vs_Parent"] > 0, "Keep candidate", "Reject candidate")
    branch["CV_Read"] = np.where(
        branch["Delta_vs_Parent"] > 0,
        "✅ beats B04 parent",
        "❌ worse than B04 parent",
    )
    keep_cols = ["Experiment", "Parent", "OOF_AUC", "Delta_vs_Parent", "Decision", "CV_Read"]
    return branch[keep_cols]


def paired_auc_delta_bootstrap(y, pred_a, pred_b, comparison="Model B - Model A", n_boot=None, seed=RANDOM_STATE):
    n_boot = 250 if FAST_MODE else 800 if n_boot is None else n_boot
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)
    n = len(y)
    deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(y[idx])) < 2:
            continue
        deltas.append(roc_auc_score(y[idx], pred_b[idx]) - roc_auc_score(y[idx], pred_a[idx]))
    deltas = np.asarray(deltas, dtype=float)
    return pd.DataFrame([
        {
            "Comparison": comparison,
            "Mean_Delta": deltas.mean(),
            "P05": np.quantile(deltas, 0.05),
            "Median": np.quantile(deltas, 0.50),
            "P95": np.quantile(deltas, 0.95),
            "Prob_Positive": np.mean(deltas > 0),
            "Bootstraps": len(deltas),
        }
    ])


def show_current_verdict(identity_summary, compressed_summary, combined_summary):
    best = combined_summary.loc[combined_summary["OOF_AUC"].idxmax()]
    raw_lift = identity_summary["OOF_AUC"].iloc[1] - identity_summary["OOF_AUC"].iloc[0]
    compressed_best = compressed_summary.loc[compressed_summary["OOF_AUC"].idxmax()]
    compressed_lift = compressed_best["OOF_AUC"] - compressed_summary["OOF_AUC"].iloc[0]

    if str(best["Experiment"]) == "A01_driver_ohe":
        headline = "Current verdict: raw Driver OHE is best, but the lift is small."
        interpretation = "Race-state features explain most of the signal. Driver identity adds a marginal correction in this saved run, so full 5-fold CV should confirm it before making a strong claim."
    elif str(best.get("Ladder", "")) == "Compressed":
        headline = "Current verdict: compressed Driver signal is the best saved-run option."
        interpretation = "A compressed Driver representation can beat raw Driver OHE in this run, but the claim still rests on OOF lift rather than feature complexity."
    else:
        headline = "Current verdict: Driver blocks do not beat the race-state foundation here."
        interpretation = "The correct conclusion is conservative: keep the race-state foundation unless a Driver block earns positive OOF lift under the same split protocol."

    html = f"""
    <div style='background:#FFFCF8; border:1px solid {COLORS['line']}; border-left:8px solid {COLORS['red']}; border-radius:16px; padding:17px 19px; margin:14px 0 22px; color:{COLORS['ink']}; box-shadow:0 10px 22px rgba(35,49,39,0.05);'>
      <div style='font-size:12px; font-weight:900; letter-spacing:.12em; color:#7A6045; text-transform:uppercase; margin-bottom:7px;'>Current CV Verdict</div>
      <div style='font-size:1.25rem; font-weight:900; line-height:1.25; margin-bottom:8px;'>{escape(headline)}</div>
      <div style='font-size:14px; line-height:1.6; color:{COLORS['muted']}; margin-bottom:13px;'>{escape(interpretation)}</div>
      <div style='display:grid; grid-template-columns:repeat(auto-fit, minmax(210px, 1fr)); gap:10px;'>
        <div style='background:#F7F3EA; border:1px solid {COLORS['line']}; border-radius:12px; padding:11px 12px;'>
          <div style='font-size:12px; font-weight:900; color:{COLORS['muted']}; text-transform:uppercase;'>Selected experiment</div>
          <div style='font-size:17px; font-weight:900; margin-top:4px;'>{escape(str(best['Experiment']))}</div>
        </div>
        <div style='background:#F7F3EA; border:1px solid {COLORS['line']}; border-radius:12px; padding:11px 12px;'>
          <div style='font-size:12px; font-weight:900; color:{COLORS['muted']}; text-transform:uppercase;'>Best OOF AUC</div>
          <div style='font-size:17px; font-weight:900; margin-top:4px;'>{best['OOF_AUC']:.5f}</div>
        </div>
        <div style='background:#F7F3EA; border:1px solid {COLORS['line']}; border-radius:12px; padding:11px 12px;'>
          <div style='font-size:12px; font-weight:900; color:{COLORS['muted']}; text-transform:uppercase;'>Raw Driver lift</div>
          <div style='font-size:17px; font-weight:900; margin-top:4px;'>{raw_lift:+.5f}</div>
        </div>
        <div style='background:#F7F3EA; border:1px solid {COLORS['line']}; border-radius:12px; padding:11px 12px;'>
          <div style='font-size:12px; font-weight:900; color:{COLORS['muted']}; text-transform:uppercase;'>Compressed best lift</div>
          <div style='font-size:17px; font-weight:900; margin-top:4px;'>{compressed_lift:+.5f}</div>
        </div>
      </div>
    </div>
    """
    display(HTML(html))


DRIVER_FOUNDATION_BLOCKS = ["race_tyre_algebra", "compound_structure", "nonlinear_interactions"]

identity_driver_configs = [
    ExperimentConfig(
        "A00_no_driver",
        "Race-state foundation without raw Driver ID",
        DRIVER_FOUNDATION_BLOCKS,
        include_driver_ohe=False,
        model_kind=LADDER_MODEL,
    ),
    ExperimentConfig(
        "A01_driver_ohe",
        "Add raw Driver one-hot identity",
        DRIVER_FOUNDATION_BLOCKS,
        include_driver_ohe=True,
        model_kind=LADDER_MODEL,
    ),
]

compressed_driver_configs = [
    ExperimentConfig(
        "B00_no_driver",
        "Race-state foundation without raw Driver ID",
        DRIVER_FOUNDATION_BLOCKS,
        include_driver_ohe=False,
        model_kind=LADDER_MODEL,
    ),
    ExperimentConfig(
        "B01_driver_string",
        "Driver string structure without raw OHE",
        DRIVER_FOUNDATION_BLOCKS + ["driver_structure"],
        include_driver_ohe=False,
        model_kind=LADDER_MODEL,
    ),
    ExperimentConfig(
        "B02_original_vocab",
        "Add original-driver vocabulary flag",
        DRIVER_FOUNDATION_BLOCKS + ["driver_structure", "driver_original_vocab"],
        include_driver_ohe=False,
        model_kind=LADDER_MODEL,
    ),
    ExperimentConfig(
        "B03_driver_frequency",
        "Add Driver/Race frequency features",
        DRIVER_FOUNDATION_BLOCKS + ["driver_structure", "driver_original_vocab", "frequency_encoding"],
        include_driver_ohe=False,
        model_kind=LADDER_MODEL,
    ),
    ExperimentConfig(
        "B04_driver_te",
        "Add inner-OOF Driver/Race target encoding",
        DRIVER_FOUNDATION_BLOCKS + ["driver_structure", "driver_original_vocab", "frequency_encoding", "oof_target_encoding"],
        include_driver_ohe=False,
        model_kind=LADDER_MODEL,
    ),
    ExperimentConfig(
        "B05_driver_compound_te",
        "Add Driver-Compound target encoding",
        DRIVER_FOUNDATION_BLOCKS + ["driver_structure", "driver_original_vocab", "frequency_encoding", "oof_target_encoding", "driver_compound_te"],
        include_driver_ohe=False,
        model_kind=LADDER_MODEL,
    ),
    ExperimentConfig(
        "B06_race_compound_te",
        "Add Race-Compound target encoding",
        DRIVER_FOUNDATION_BLOCKS + ["driver_structure", "driver_original_vocab", "frequency_encoding", "oof_target_encoding", "race_compound_te"],
        include_driver_ohe=False,
        model_kind=LADDER_MODEL,
    ),
    ExperimentConfig(
        "B07_race_stint_te",
        "Add Race-Stint target encoding",
        DRIVER_FOUNDATION_BLOCKS + ["driver_structure", "driver_original_vocab", "frequency_encoding", "oof_target_encoding", "race_stint_te"],
        include_driver_ohe=False,
        model_kind=LADDER_MODEL,
    ),
    ExperimentConfig(
        "B08_compound_stint_te",
        "Add Compound-Stint target encoding",
        DRIVER_FOUNDATION_BLOCKS + ["driver_structure", "driver_original_vocab", "frequency_encoding", "oof_target_encoding", "compound_stint_te"],
        include_driver_ohe=False,
        model_kind=LADDER_MODEL,
    ),
]

identity_driver_results = [run_oof_experiment(cfg, verbose=True) for cfg in identity_driver_configs]
identity_driver_summary = summarize_results(identity_driver_results, baseline_auc=identity_driver_results[0]["auc"])
identity_driver_summary = enrich_ladder_summary(identity_driver_summary)

display_table(identity_driver_summary, "Driver identity ladder: how much does raw one-hot ID add?")
plot_driver_ladder(identity_driver_summary, "Raw Driver identity: no Driver vs one-hot")

driver_ohe_bootstrap = paired_auc_delta_bootstrap(
    cv_train[TARGET].to_numpy(),
    identity_driver_results[0]["oof"],
    identity_driver_results[1]["oof"],
    comparison="A01 Driver OHE - A00 No Driver",
)
display_table(driver_ohe_bootstrap, "Paired bootstrap check for the tiny Driver OHE delta")

compressed_driver_results = [run_oof_experiment(cfg, verbose=True) for cfg in compressed_driver_configs]
compressed_driver_summary = summarize_results(compressed_driver_results, baseline_auc=compressed_driver_results[0]["auc"])
compressed_driver_summary = enrich_ladder_summary(compressed_driver_summary)

show_driver_takeaway(compressed_driver_summary)
display_table(compressed_driver_summary, "Compressed Driver ladder: structure, frequency, and fold-safe target encodings")

interaction_branch_table = make_interaction_branch_table(compressed_driver_summary)
if not interaction_branch_table.empty:
    display_table(interaction_branch_table, "Interaction TE branch tests: each candidate compared against B04 parent")

compressed_best_idx = int(np.argmax([result["auc"] for result in compressed_driver_results]))
compressed_best_result = compressed_driver_results[compressed_best_idx]
compressed_bootstrap = paired_auc_delta_bootstrap(
    cv_train[TARGET].to_numpy(),
    compressed_driver_results[0]["oof"],
    compressed_best_result["oof"],
    comparison=f"{compressed_best_result['config'].name} - B00 No Driver",
)
display_table(compressed_bootstrap, "Paired bootstrap check for the best compressed Driver delta")

plot_driver_ladder(compressed_driver_summary, "Compressed Driver signal: raw step vs cumulative best")
show_driver_argument(identity_driver_summary, compressed_driver_summary)

driver_ladder_results = identity_driver_results + compressed_driver_results
combined_driver_summary = summarize_results(driver_ladder_results, baseline_auc=identity_driver_results[0]["auc"])
combined_driver_summary["Ladder"] = ["Raw OHE"] * len(identity_driver_results) + ["Compressed"] * len(compressed_driver_results)
driver_ladder_summary = combined_driver_summary
display_table(combined_driver_summary, "Driver ladder comparison: raw identity vs compressed signal")
show_current_verdict(identity_driver_summary, compressed_driver_summary, combined_driver_summary)

best_driver_result = max(driver_ladder_results, key=lambda result: result["auc"])
best_driver_config = best_driver_result["config"]
print(f"Best Driver ladder experiment: {best_driver_config.name} | OOF AUC {best_driver_result['auc']:.5f}")

if RUN_FULL_FEATURE_LADDER:
    ladder_configs = [
        ExperimentConfig("E00_no_driver", "Raw features without Driver", [], include_driver_ohe=False, model_kind=LADDER_MODEL),
        ExperimentConfig("E01_driver_ohe", "Add Driver one-hot", [], include_driver_ohe=True, model_kind=LADDER_MODEL),
        ExperimentConfig("E02_race_tyre", "Race/tyre algebra", ["race_tyre_algebra"], True, LADDER_MODEL),
        ExperimentConfig("E03_compound", "Compound structure", ["race_tyre_algebra", "compound_structure"], True, LADDER_MODEL),
        ExperimentConfig("E04_driver_structure", "Driver string structure", ["race_tyre_algebra", "compound_structure", "driver_structure"], True, LADDER_MODEL),
        ExperimentConfig("E05_frequency", "Frequency/count encoding", ["race_tyre_algebra", "compound_structure", "driver_structure", "driver_original_vocab", "frequency_encoding"], True, LADDER_MODEL),
        ExperimentConfig("E06_context", "Race-context normalization", ["race_tyre_algebra", "compound_structure", "driver_structure", "driver_original_vocab", "frequency_encoding", "context_normalization"], True, LADDER_MODEL),
        ExperimentConfig("E07_interactions", "Nonlinear interactions", ["race_tyre_algebra", "compound_structure", "driver_structure", "driver_original_vocab", "frequency_encoding", "context_normalization", "nonlinear_interactions"], True, LADDER_MODEL),
        ExperimentConfig("E08_target_encoding", "OOF Driver/Race target encoding", ["race_tyre_algebra", "compound_structure", "driver_structure", "driver_original_vocab", "frequency_encoding", "context_normalization", "nonlinear_interactions", "oof_target_encoding"], True, LADDER_MODEL),
        ExperimentConfig("E09_interaction_te", "OOF interaction target encoding", ["race_tyre_algebra", "compound_structure", "driver_structure", "driver_original_vocab", "frequency_encoding", "context_normalization", "nonlinear_interactions", "oof_target_encoding", "interaction_target_encoding"], True, LADDER_MODEL),
    ]
    ladder_results = [run_oof_experiment(cfg, verbose=True) for cfg in ladder_configs]
    ladder_summary = summarize_results(ladder_results, baseline_auc=ladder_results[0]["auc"])
    ladder_summary = enrich_ladder_summary(ladder_summary)
    display_table(ladder_summary, "Full feature ladder: fixed model")
    plot_ladder(ladder_summary, "Full feature ladder OOF AUC")
    combined_candidates = ladder_results + [best_driver_result]
    combined_summary = summarize_results(combined_candidates, baseline_auc=ladder_results[0]["auc"])
    best_ladder_idx = int(combined_summary["OOF_AUC"].idxmax())
    best_ladder_result = combined_candidates[best_ladder_idx]
else:
    print("Full feature ladder skipped. The selected setup comes from the Driver-centered ladder.")
    ladder_results = driver_ladder_results
    ladder_summary = driver_ladder_summary
    best_ladder_result = best_driver_result

best_ladder_config = best_ladder_result["config"]
print(f"Selected ladder experiment: {best_ladder_config.name} | OOF AUC {best_ladder_result['auc']:.5f}")

## 7. Score-Oriented Candidate Stack

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-left:6px solid #B5473C; border-radius:14px; padding:13px 15px; margin:10px 0 16px 0; color:#233127; line-height:1.58; box-shadow:0 8px 18px rgba(35,49,39,0.04);">The Driver ladder is still the spine. This section asks a narrower follow-up: do public-domain features or the preprocessing style from the high-score stack beat the current Driver best under the same saved-run CV?</div>


In [ ]:
SCORE_FOUNDATION_BLOCKS = [
    "race_tyre_algebra",
    "compound_structure",
    "public_domain_features",
    "nonlinear_interactions",
]
SCORE_SAFE_DRIVER_BLOCKS = SCORE_FOUNDATION_BLOCKS + [
    "driver_structure",
    "driver_original_vocab",
    "frequency_encoding",
    "context_normalization",
]

score_candidate_configs = [
    ExperimentConfig(
        "S00_driver_ohe_domain",
        "Driver OHE + public pit-window domain features",
        SCORE_FOUNDATION_BLOCKS,
        include_driver_ohe=True,
        model_kind=LADDER_MODEL,
    ),
    ExperimentConfig(
        "S01_driver_ohe_context",
        "Add weighted support and field-relative context",
        SCORE_SAFE_DRIVER_BLOCKS,
        include_driver_ohe=True,
        model_kind=LADDER_MODEL,
    ),
    ExperimentConfig(
        "S02_driver_ohe_driver_te",
        "Add inner-OOF Driver/Race target encoding",
        SCORE_SAFE_DRIVER_BLOCKS + ["oof_target_encoding"],
        include_driver_ohe=True,
        model_kind=LADDER_MODEL,
    ),
    ExperimentConfig(
        "S03_driver_ohe_race_compound_te",
        "Test Race-Compound TE on top of the score stack",
        SCORE_SAFE_DRIVER_BLOCKS + ["oof_target_encoding", "race_compound_te"],
        include_driver_ohe=True,
        model_kind=LADDER_MODEL,
    ),
    ExperimentConfig(
        "S04_driver_ohe_driver_compound_te",
        "Test Driver-Compound TE on top of the score stack",
        SCORE_SAFE_DRIVER_BLOCKS + ["oof_target_encoding", "driver_compound_te"],
        include_driver_ohe=True,
        model_kind=LADDER_MODEL,
    ),
]

STACK_KEY_FREQUENCY_COLUMNS = ["Driver", "Race", "Compound", "Race_Compound", "Driver_Compound", "Race_Stint", "Compound_Stint"]
STACK_MODEL_KIND = "lgbm" if lgb is not None else LADDER_MODEL


def add_stack_frequency_features(train_base, test_base):
    train_base = train_base.copy()
    test_base = test_base.copy()
    train_base["__part"] = "train"
    test_base["__part"] = "test"
    train_base["__row"] = np.arange(len(train_base))
    test_base["__row"] = np.arange(len(test_base))
    full = pd.concat([train_base, test_base], ignore_index=True, sort=False)
    for col in STACK_KEY_FREQUENCY_COLUMNS:
        if col not in full.columns:
            continue
        freq = full[col].astype(str).value_counts(dropna=False)
        full[f"{col}_freq"] = full[col].astype(str).map(freq).fillna(0).astype(float)
        full[f"{col}_freq_log1p"] = np.log1p(full[f"{col}_freq"])
    train_out = full[full["__part"].eq("train")].sort_values("__row").drop(columns=["__part", "__row"])
    test_out = full[full["__part"].eq("test")].sort_values("__row").drop(columns=["__part", "__row", TARGET], errors="ignore")
    return train_out.reset_index(drop=True), test_out.reset_index(drop=True)


def add_stack_core_features(train_df, test_df):
    train_base = add_base_interactions(train_df)
    test_base = add_base_interactions(test_df)
    return add_stack_frequency_features(train_base, test_base)


def add_stack_row_features(df):
    out = add_base_interactions(df)
    out = add_race_tyre_algebra(out)
    out = add_compound_structure(out)
    out = add_public_domain_features(out)
    out = add_driver_structure(out)
    out = add_nonlinear_interactions(out)
    return out


def add_stack_domain_features(train_df, test_df):
    train_base = add_stack_row_features(train_df)
    test_base = add_stack_row_features(test_df)
    train_base["__part"] = "train"
    test_base["__part"] = "test"
    train_base["__row"] = np.arange(len(train_base))
    test_base["__row"] = np.arange(len(test_base))
    full = pd.concat([train_base, test_base], ignore_index=True, sort=False)

    sort_cols = [col for col in ["Race", "Driver", "Year", "Stint", "LapNumber"] if col in full.columns]
    full = full.sort_values(sort_cols).copy()
    group_cols = [col for col in ["Race", "Driver", "Year", "Stint"] if col in full.columns]
    g = full.groupby(group_cols, sort=False, observed=True)

    if "LapNumber" in full.columns:
        full["stack_lap_in_stint"] = g.cumcount() + 1
        full["stack_stint_len"] = g["LapNumber"].transform("count")
        full["stack_stint_min_lap"] = g["LapNumber"].transform("min")
        full["stack_stint_max_lap"] = g["LapNumber"].transform("max")
        full["stack_circuit_max_lap"] = full.groupby("Race", observed=True)["LapNumber"].transform("max")
        full["stack_lap_frac_circuit"] = _safe_divide(full["LapNumber"].astype(float), full["stack_circuit_max_lap"].astype(float))

    if "Position" in full.columns:
        full["stack_pos_prev1"] = g["Position"].shift(1)
        full["stack_pos_prev3"] = g["Position"].shift(3)
        full["stack_pos_delta1"] = full["Position"] - full["stack_pos_prev1"]
        full["stack_pos_delta3"] = full["Position"] - full["stack_pos_prev3"]
        full["stack_pos_min_so_far"] = g["Position"].cummin()
        full["stack_pos_max_so_far"] = g["Position"].cummax()

    if "LapTime (s)" in full.columns:
        full["stack_lt_prev1"] = g["LapTime (s)"].shift(1)
        full["stack_lt_prev2"] = g["LapTime (s)"].shift(2)
        full["stack_lt_delta1"] = full["LapTime (s)"] - full["stack_lt_prev1"]
        full["stack_lt_roll3_mean"] = g["LapTime (s)"].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
        full["stack_lt_vs_roll3"] = full["LapTime (s)"] - full["stack_lt_roll3_mean"]

    if "TyreLife" in full.columns:
        full["stack_tyre_life_lag1"] = g["TyreLife"].shift(1)
        full["stack_tyre_life_diff1"] = full["TyreLife"] - full["stack_tyre_life_lag1"]
        full["stack_race_lap_meantyrelife"] = full.groupby(["Race", "LapNumber"], observed=True)["TyreLife"].transform("mean")
        full["stack_race_lap_maxtyrelife"] = full.groupby(["Race", "LapNumber"], observed=True)["TyreLife"].transform("max")
        full["stack_tyrelife_vs_field_mean"] = full["TyreLife"] - full["stack_race_lap_meantyrelife"]
        full["stack_compound_meantyrelife"] = full.groupby("Compound", observed=True)["TyreLife"].transform("mean")
        full["stack_compound_stdtyrelife"] = full.groupby("Compound", observed=True)["TyreLife"].transform("std")
        full["stack_tyrelife_vs_compound"] = full["TyreLife"] - full["stack_compound_meantyrelife"]

    for col in STACK_KEY_FREQUENCY_COLUMNS:
        if col not in full.columns:
            continue
        freq = full[col].astype(str).value_counts(dropna=False)
        full[f"stack_{col}_freq"] = full[col].astype(str).map(freq).fillna(0).astype(float)
        full[f"stack_{col}_freq_log1p"] = np.log1p(full[f"stack_{col}_freq"])

    full = full.sort_values(["__part", "__row"]).copy()
    train_out = full[full["__part"].eq("train")].sort_values("__row").drop(columns=["__part", "__row"])
    test_out = full[full["__part"].eq("test")].sort_values("__row").drop(columns=["__part", "__row", TARGET], errors="ignore")
    return train_out.reset_index(drop=True), test_out.reset_index(drop=True)


def build_stack_preprocessing_matrix(train_df, test_df, feature_set):
    if feature_set == "core":
        train_features, test_features = add_stack_core_features(train_df, test_df)
    elif feature_set == "domain":
        train_features, test_features = add_stack_domain_features(train_df, test_df)
    else:
        raise ValueError(f"Unknown stack preprocessing feature set: {feature_set}")

    drop_cols = {TARGET, ID_COL}
    feature_cols = [col for col in train_features.columns if col not in drop_cols]
    X = train_features[feature_cols].copy()
    X_test = test_features[feature_cols].copy()
    cat_cols = [col for col in feature_cols if X[col].dtype == "object" or str(X[col].dtype).startswith("category")]

    for col in cat_cols:
        combined = pd.concat([X[col], X_test[col]], axis=0).astype(str).fillna("MISSING")
        codes, _ = pd.factorize(combined, sort=True)
        X[col] = codes[:len(X)].astype(np.int32)
        X_test[col] = codes[len(X):].astype(np.int32)

    X = X.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)
    return X, X_test, feature_cols, cat_cols


def run_stack_preprocessing_experiment(name, label, feature_set, model_kind=None):
    model_kind = model_kind or STACK_MODEL_KIND
    y = cv_train[TARGET].astype(int).to_numpy()
    X, X_test, feature_cols, cat_cols = build_stack_preprocessing_matrix(cv_train, test, feature_set)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(cv_train), dtype=float)
    test_pred = np.zeros(len(test), dtype=float)
    fold_rows = []

    for fold, (fit_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        model = make_model(model_kind)
        valid_pred, fold_test_pred, fitted_model = fit_predict_model(
            model_kind,
            model,
            X.iloc[fit_idx],
            y[fit_idx],
            X.iloc[valid_idx],
            y[valid_idx],
            X_test,
            sample_weight=None,
        )
        oof[valid_idx] = valid_pred
        test_pred += fold_test_pred / N_SPLITS
        fold_auc = roc_auc_score(y[valid_idx], valid_pred)
        fold_rows.append({
            "Fold": fold,
            "AUC": fold_auc,
            "Features": len(feature_cols),
            "Ordinal_Categorical_Features": len(cat_cols),
        })
        print(f"{name} | fold {fold}/{N_SPLITS} | AUC {fold_auc:.5f}")

    return {
        "config": ExperimentConfig(
            name,
            label,
            blocks=[f"stack_preprocessing_{feature_set}"],
            include_driver_ohe=False,
            model_kind=f"{model_kind}_ordinal_{feature_set}",
            original_weight=0.0,
        ),
        "oof": oof,
        "test_pred": test_pred,
        "folds": pd.DataFrame(fold_rows),
        "auc": roc_auc_score(y, oof),
        "logloss": log_loss(y, np.clip(oof, 1e-6, 1 - 1e-6)),
        "models": [],
        "numeric_features": feature_cols,
        "categorical_features": cat_cols,
    }


if RUN_SCORE_STACK:
    score_candidate_results = [run_oof_experiment(cfg, verbose=True) for cfg in score_candidate_configs]
    stack_preprocessing_results = [
        run_stack_preprocessing_experiment(
            "S05_stack_core_preprocessing",
            "High-score stack core preprocessing: ordinal + frequency matrix",
            "core",
        ),
        run_stack_preprocessing_experiment(
            "S06_stack_domain_preprocessing",
            "High-score stack domain preprocessing: sequence/group matrix",
            "domain",
        ),
    ]
    stack_matrix_summary = pd.DataFrame([
        {
            "Experiment": result["config"].name,
            "Feature_Count": len(result["numeric_features"]),
            "Ordinal_Categorical_Features": len(result["categorical_features"]),
            "OOF_AUC": result["auc"],
            "Delta_vs_Current_Best": result["auc"] - best_ladder_result["auc"],
        }
        for result in stack_preprocessing_results
    ])
    display_table(stack_matrix_summary, "Stack preprocessing candidates imported from the high-score notebook")

    score_candidate_results = score_candidate_results + stack_preprocessing_results
    score_candidate_summary = summarize_results(score_candidate_results, baseline_auc=best_ladder_result["auc"])
    score_candidate_summary = enrich_ladder_summary(score_candidate_summary)
    score_candidate_summary["Delta_vs_Current_Best"] = score_candidate_summary["OOF_AUC"] - best_ladder_result["auc"]
    score_candidate_summary["Decision"] = np.where(
        score_candidate_summary["Delta_vs_Current_Best"] > 0,
        "Candidate improves current OOF best",
        "Keep previous best",
    )
    display_table(score_candidate_summary, "Score candidate stack: public-domain and stack-preprocessing candidates")
    plot_ladder(score_candidate_summary, "Score candidate stack OOF AUC")

    score_pool = [best_ladder_result] + score_candidate_results
    score_pool_summary = summarize_results(score_pool, baseline_auc=best_ladder_result["auc"])
    best_score_idx = int(score_pool_summary["OOF_AUC"].idxmax())
    best_ladder_result = score_pool[best_score_idx]
    best_ladder_config = best_ladder_result["config"]
else:
    print("Score candidate stack skipped. The selected setup comes from the Driver-centered ladder.")
    score_candidate_results = []
    stack_preprocessing_results = []
    score_candidate_summary = pd.DataFrame()

print(f"Selected score setup: {best_ladder_config.name} | OOF AUC {best_ladder_result['auc']:.5f}")


## 8. F1 Strategy Feature Diagnostic

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-left:6px solid #3D6F8A; border-radius:14px; padding:13px 15px; margin:10px 0 16px 0; color:#233127; line-height:1.58; box-shadow:0 8px 18px rgba(35,49,39,0.04);">F1 domain knowledge is useful only when it is expressed in a form the synthetic tabular problem actually rewards. This diagnostic separates naive physics priors from field-relative and sequence-relative strategy features. The result is intentionally conservative: simple physical priors are rejected; relative race-state features are plausible, but the saved diagnostic does not yet justify promoting them above the current Driver-frequency stack.</div>


In [ ]:
f1_strategy_diag_strict = pd.DataFrame([
    {"Protocol": "Fold-fitted", "Sample_Rows": 40_000, "Experiment": "E0 raw + freq", "OOF_AUC": 0.932083, "Delta_vs_E0": 0.000000, "Feature_Count": 19},
    {"Protocol": "Fold-fitted", "Sample_Rows": 40_000, "Experiment": "E1 F1 core priors", "OOF_AUC": 0.931096, "Delta_vs_E0": -0.000987, "Feature_Count": 40},
    {"Protocol": "Fold-fitted", "Sample_Rows": 40_000, "Experiment": "E2 F1 relative state", "OOF_AUC": 0.929100, "Delta_vs_E0": -0.002983, "Feature_Count": 55},
    {"Protocol": "Fold-fitted", "Sample_Rows": 80_000, "Experiment": "E0 raw + freq", "OOF_AUC": 0.939447, "Delta_vs_E0": 0.000000, "Feature_Count": 19},
    {"Protocol": "Fold-fitted", "Sample_Rows": 80_000, "Experiment": "E2 F1 relative state", "OOF_AUC": 0.938592, "Delta_vs_E0": -0.000856, "Feature_Count": 55},
])

f1_strategy_diag_transductive = pd.DataFrame([
    {"Protocol": "Unlabeled row-structure", "Sample_Rows": 40_000, "Experiment": "E0 raw + freq", "OOF_AUC": 0.932188, "Delta_vs_E0": 0.000000, "Feature_Count": 19},
    {"Protocol": "Unlabeled row-structure", "Sample_Rows": 40_000, "Experiment": "E1 F1 core priors", "OOF_AUC": 0.931346, "Delta_vs_E0": -0.000842, "Feature_Count": 40},
    {"Protocol": "Unlabeled row-structure", "Sample_Rows": 40_000, "Experiment": "E2 F1 relative state", "OOF_AUC": 0.931571, "Delta_vs_E0": -0.000617, "Feature_Count": 55},
    {"Protocol": "Unlabeled row-structure", "Sample_Rows": 80_000, "Experiment": "E0 raw + freq", "OOF_AUC": 0.939763, "Delta_vs_E0": 0.000000, "Feature_Count": 19},
    {"Protocol": "Unlabeled row-structure", "Sample_Rows": 80_000, "Experiment": "E2 F1 relative state", "OOF_AUC": 0.939848, "Delta_vs_E0": 0.000085, "Feature_Count": 55},
])

f1_strategy_diag = pd.concat([f1_strategy_diag_strict, f1_strategy_diag_transductive], ignore_index=True)

display_cards([
    metric_card("Naive F1 priors", "reject", "E1 is below E0 in the 40k diagnostic", COLORS["red"]),
    metric_card("Strict relative features", "not promoted", "E2 is below E0 with fold-fitted stats", COLORS["gold"]),
    metric_card("Transductive relative features", "+0.00009", "80k diagnostic only; plausible but tiny", COLORS["blue"]),
    metric_card("Modeling read", "candidate", "test inside heavy GBDT stack, not as Driver replacement", COLORS["green"]),
], columns=4)

display_table(f1_strategy_diag, "F1 strategy feature diagnostic: strict vs unlabeled row-structure protocols")

fig, ax = plt.subplots(figsize=(11.5, 4.8))
plot_df = f1_strategy_diag[f1_strategy_diag["Experiment"].ne("E0 raw + freq")].copy()
plot_df["Label"] = plot_df["Protocol"] + " | " + plot_df["Sample_Rows"].map(lambda x: f"{int(x/1000)}k") + " | " + plot_df["Experiment"]
colors = np.where(plot_df["Delta_vs_E0"] > 0, COLORS["green"], COLORS["red"])
ax.barh(plot_df["Label"], plot_df["Delta_vs_E0"], color=colors, alpha=0.85)
ax.axvline(0, color=COLORS["ink"], linewidth=1.2)
for value, label in zip(plot_df["Delta_vs_E0"], plot_df["Label"]):
    ha = "left" if value >= 0 else "right"
    x = value + (0.00008 if value >= 0 else -0.00008)
    ax.text(x, label, f"{value:+.6f}", va="center", ha=ha, fontsize=9.5, fontweight="bold", color=COLORS["ink"])
ax.set_title("F1 feature block deltas: physical priors alone are not enough", fontsize=13.5, fontweight="bold", color=COLORS["ink"])
ax.set_xlabel("OOF AUC delta vs E0 raw + frequency baseline")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

f1_strategy_importance = pd.DataFrame([
    {"Rank": 1, "Feature": "degradation_stress", "Strategic_Read": "pace collapse pressure: abs lap-time delta multiplied by tyre age"},
    {"Rank": 2, "Feature": "Stint", "Strategic_Read": "pit-cycle stage remains a dominant race-state signal"},
    {"Rank": 3, "Feature": "estimated_total_laps", "Strategic_Read": "race-length and remaining-race context"},
    {"Rank": 4, "Feature": "TyreLife_dev_by_raceyearlap", "Strategic_Read": "field-relative tyre age at the same race/year/lap"},
    {"Rank": 5, "Feature": "stint_x_tyre_life", "Strategic_Read": "stint-cycle pressure: later stint and older tyre"},
    {"Rank": 7, "Feature": "tyre_life_x_hardness", "Strategic_Read": "compound-adjusted tyre age"},
    {"Rank": 10, "Feature": "estimated_laps_remaining", "Strategic_Read": "phase of the race from the remaining-lap view"},
    {"Rank": 18, "Feature": "traffic_pressure_proxy", "Strategic_Read": "position pressure proxy for undercut/traffic incentive"},
])
display_table(f1_strategy_importance, "F1-informed features that the diagnostic model still used heavily")

f1_strategy_read = pd.DataFrame([
    {"Finding": "Simple physical priors are fragile", "Implication": "Do not hard-code SOFT/MEDIUM/HARD assumptions as if this were real telemetry."},
    {"Finding": "Relative state is more promising", "Implication": "Features such as field-relative tyre age and stint-relative pace volatility are closer to real pit strategy logic."},
    {"Finding": "Current diagnostic gain is not enough", "Implication": "Keep this as a candidate for XGB/Cat/LGBM high-score stacks, not as a replacement for the selected Driver-frequency model."},
])
display_table(f1_strategy_read, "Decision from the F1 strategy diagnostic")


## 9. Leaderboard Pivot: Driver Signal vs Score Strategy

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-left:6px solid #B5473C; border-radius:14px; padding:13px 15px; margin:10px 0 16px 0; color:#233127; line-height:1.58; box-shadow:0 8px 18px rgba(35,49,39,0.04);">The Driver ladder is useful because it tells us what survives. It is not, by itself, the strongest leaderboard strategy. This section makes the pivot explicit: Driver frequency is treated as a small auxiliary model, while the largest score gap points toward original-data-augmented XGBoost/CatBoost/LGBM stacks and OOF rank blending.</div>


In [ ]:
PUBLIC_REFERENCE_STACKS = pd.DataFrame([
    {
        "Model_or_Stack": "Domain LGBM public reference",
        "Reference_OOF_AUC": 0.94812,
        "Role": "domain-feature baseline",
    },
    {
        "Model_or_Stack": "RealMLP public reference",
        "Reference_OOF_AUC": 0.94951,
        "Role": "model-family diversity",
    },
    {
        "Model_or_Stack": "XGB baseline + original",
        "Reference_OOF_AUC": 0.95816,
        "Role": "original-data GBDT anchor",
    },
    {
        "Model_or_Stack": "XGB Optuna + original",
        "Reference_OOF_AUC": 0.95994,
        "Role": "single-model heavy anchor",
    },
    {
        "Model_or_Stack": "XGB + CatBoost ensemble",
        "Reference_OOF_AUC": 0.96076,
        "Role": "public high-score anchor",
    },
])

current_driver_auc = float(best_ladder_result["auc"])
score_best_auc = np.nan
score_best_name = "not run"
if score_candidate_results:
    score_best_result = max(score_candidate_results, key=lambda result: result["auc"])
    score_best_auc = float(score_best_result["auc"])
    score_best_name = score_best_result["config"].name

pivot_rows = [
    {
        "Model_or_Stack": f"Selected Driver stack: {best_ladder_config.name}",
        "Reference_OOF_AUC": current_driver_auc,
        "Role": "current notebook selection",
    }
]
if np.isfinite(score_best_auc):
    pivot_rows.append({
        "Model_or_Stack": f"Best score candidate: {score_best_name}",
        "Reference_OOF_AUC": score_best_auc,
        "Role": "current notebook candidate",
    })

score_gap_table = pd.concat([pd.DataFrame(pivot_rows), PUBLIC_REFERENCE_STACKS], ignore_index=True)
score_gap_table["Gap_vs_Current_Driver"] = score_gap_table["Reference_OOF_AUC"] - current_driver_auc
score_gap_table["Read"] = np.where(
    score_gap_table["Gap_vs_Current_Driver"] > 0.005,
    "leaderboard-scale gap",
    np.where(score_gap_table["Gap_vs_Current_Driver"] > 0, "small improvement", "current local best"),
)

display_cards([
    metric_card("Selected Driver OOF", f"{current_driver_auc:.5f}", best_ladder_config.name, COLORS["green"]),
    metric_card("Best public reference", f"{PUBLIC_REFERENCE_STACKS['Reference_OOF_AUC'].max():.5f}", "XGB + CatBoost ensemble", COLORS["red"]),
    metric_card("Score gap", f"{PUBLIC_REFERENCE_STACKS['Reference_OOF_AUC'].max() - current_driver_auc:+.5f}", "public anchor minus current Driver stack", COLORS["gold"]),
    metric_card("Practical role", "auxiliary", "Driver belongs as a small diversity signal", COLORS["blue"]),
], columns=4)

display_table(score_gap_table, "Score gap chart data: Driver basis points vs leaderboard-grade GBDT")

plot_df = score_gap_table.sort_values("Reference_OOF_AUC", ascending=True).copy()
fig, ax = plt.subplots(figsize=(11.5, 5.8))
colors = [COLORS["green"] if "Selected Driver" in name else COLORS["blue"] if "score candidate" in name else COLORS["red"] for name in plot_df["Model_or_Stack"]]
ax.barh(plot_df["Model_or_Stack"], plot_df["Reference_OOF_AUC"], color=colors, alpha=0.88)
for value, label in zip(plot_df["Reference_OOF_AUC"], plot_df["Model_or_Stack"]):
    ax.text(value + 0.00025, label, f"{value:.5f}", va="center", ha="left", fontsize=9.5, fontweight="bold", color=COLORS["ink"])
ax.axvline(current_driver_auc, color=COLORS["green"], linestyle="--", linewidth=1.8, label="selected Driver stack")
ax.set_xlabel("OOF ROC-AUC")
ax.set_ylabel("")
ax.set_title("Leaderboard pivot: Driver engineering gives basis points; original + heavy GBDT gives percentage points", fontsize=13.5, fontweight="bold", color=COLORS["ink"])
ax.set_xlim(max(0.5, plot_df["Reference_OOF_AUC"].min() - 0.004), min(1.0, plot_df["Reference_OOF_AUC"].max() + 0.004))
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.35)
plt.tight_layout()
plt.show()


def _auc_from_summary(summary, experiment):
    if summary is None or summary.empty or experiment not in set(summary["Experiment"]):
        return np.nan
    return float(summary.loc[summary["Experiment"].eq(experiment), "OOF_AUC"].iloc[0])


def _decision(delta, keep_label="Keep candidate", reject_label="Reject"):
    if not np.isfinite(delta):
        return "Not tested"
    return keep_label if delta > 0 else reject_label

b00 = _auc_from_summary(compressed_driver_summary, "B00_no_driver")
b01 = _auc_from_summary(compressed_driver_summary, "B01_driver_string")
b03 = _auc_from_summary(compressed_driver_summary, "B03_driver_frequency")
b04 = _auc_from_summary(compressed_driver_summary, "B04_driver_te")
a00 = _auc_from_summary(identity_driver_summary, "A00_no_driver")
a01 = _auc_from_summary(identity_driver_summary, "A01_driver_ohe")

accepted_rows = [
    {
        "Feature_Block": "Raw Driver OHE",
        "Decision": _decision(a01 - a00, "Small candidate, not automatically final"),
        "Reason": f"A01 - A00 = {a01 - a00:+.5f} AUC",
    },
    {
        "Feature_Block": "Driver string structure",
        "Decision": _decision(b01 - b00),
        "Reason": f"B01 - B00 = {b01 - b00:+.5f} AUC",
    },
    {
        "Feature_Block": "Driver frequency/support",
        "Decision": "Keep" if best_ladder_config.name == "B03_driver_frequency" else _decision(b03 - b00),
        "Reason": f"B03 - B00 = {b03 - b00:+.5f} AUC",
    },
    {
        "Feature_Block": "Driver/Race target encoding",
        "Decision": _decision(b04 - b03),
        "Reason": f"B04 - B03 = {b04 - b03:+.5f} AUC",
    },
]

if 'interaction_branch_table' in globals() and not interaction_branch_table.empty:
    for _, row in interaction_branch_table.iterrows():
        accepted_rows.append({
            "Feature_Block": str(row["Experiment"]).replace("B05_", "").replace("B06_", "").replace("B07_", "").replace("B08_", ""),
            "Decision": row["Decision"],
            "Reason": f"vs B04 parent = {row['Delta_vs_Parent']:+.5f} AUC",
        })

if np.isfinite(score_best_auc):
    accepted_rows.append({
        "Feature_Block": "Public pit-window score stack",
        "Decision": _decision(score_best_auc - current_driver_auc, "Promote", "Reject in this LGBM stack"),
        "Reason": f"best score candidate - selected Driver = {score_best_auc - current_driver_auc:+.5f} AUC",
    })

if 'original_summary' in globals() and isinstance(original_summary, pd.DataFrame) and not original_summary.empty:
    original_best = float(original_summary["OOF_AUC"].max())
    accepted_rows.append({
        "Feature_Block": "Original data ablation",
        "Decision": _decision(original_best - current_driver_auc, "Promote best weight", "Do not promote"),
        "Reason": f"best original weight - selected Driver = {original_best - current_driver_auc:+.5f} AUC",
    })
else:
    accepted_rows.append({
        "Feature_Block": "Original data ablation",
        "Decision": "Not tested in saved output",
        "Reason": "Enable RUN_ORIGINAL_ABLATION for the score run",
    })

accepted_rows.append({
    "Feature_Block": "XGB/CatBoost public anchor",
    "Decision": "Strong score candidate",
    "Reason": f"public reference gap = {PUBLIC_REFERENCE_STACKS['Reference_OOF_AUC'].max() - current_driver_auc:+.5f} AUC",
})

accepted_rejected_table = pd.DataFrame(accepted_rows)
display_table(accepted_rejected_table, "Accepted vs rejected feature/model blocks")


## 10. Driver Residual Correction Layer

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-left:6px solid #2F6B4F; border-radius:14px; padding:13px 15px; margin:10px 0 16px 0; color:#233127; line-height:1.58; box-shadow:0 8px 18px rgba(35,49,39,0.04);">The second half changes the Driver question. Driver no longer has to be the main predictor; it only has to correct systematic underprediction or overprediction left by the selected stack. The test is strict: estimate fold-safe Driver logit bias, add it to the base logit, and promote it only if OOF AUC improves.</div>


In [ ]:
def sigmoid_array(x):
    x = np.asarray(x, dtype=float)
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))


def logit_clip(p, eps=1e-6):
    p = np.clip(np.asarray(p, dtype=float), eps, 1 - eps)
    return np.log(p / (1 - p))


def build_residual_frame(frame):
    out = pd.DataFrame(index=frame.index)
    driver = frame["Driver"].astype(str).fillna("MISSING")
    compound = frame["Compound"].astype(str).fillna("MISSING")
    race = frame["Race"].astype(str).fillna("MISSING")
    stint = frame["Stint"].astype("Int64").astype(str).replace("<NA>", "MISSING")
    phase = pd.cut(
        frame["RaceProgress"].astype(float),
        bins=[-0.01, 0.25, 0.65, 1.05],
        labels=["early", "mid", "late"],
    ).astype(str)

    out["Driver"] = driver
    out["Driver_Phase"] = driver + "__" + phase
    out["Driver_Compound"] = driver + "__" + compound
    out["Driver_Stint"] = driver + "__" + stint
    out["Driver_Race"] = driver + "__" + race
    return out


def fit_logit_residual_maps(residual_frame, y, base_pred, group_alphas):
    maps = {}
    y = np.asarray(y, dtype=float)
    base_pred = np.clip(np.asarray(base_pred, dtype=float), 1e-6, 1 - 1e-6)
    global_y = float(np.mean(y))
    global_p = float(np.mean(base_pred))

    for col, alpha in group_alphas.items():
        tmp = pd.DataFrame({
            "key": residual_frame[col].astype(str),
            "y": y,
            "p": base_pred,
        })
        grouped = tmp.groupby("key", observed=True).agg(
            n=("y", "size"),
            y_sum=("y", "sum"),
            p_sum=("p", "sum"),
        )
        grouped["y_smooth"] = (grouped["y_sum"] + alpha * global_y) / (grouped["n"] + alpha)
        grouped["p_smooth"] = (grouped["p_sum"] + alpha * global_p) / (grouped["n"] + alpha)
        grouped["delta_logit"] = logit_clip(grouped["y_smooth"]) - logit_clip(grouped["p_smooth"])
        maps[col] = grouped[["delta_logit", "n"]]
    return maps


def apply_logit_residual_signal(residual_frame, maps, group_weights):
    signal = np.zeros(len(residual_frame), dtype=float)
    for col, weight in group_weights.items():
        stat = maps[col]
        delta = residual_frame[col].astype(str).map(stat["delta_logit"]).fillna(0.0).astype(float).to_numpy()
        signal += weight * delta
    return signal


RESIDUAL_PROMOTION_MIN_AUC = 3e-4


def residual_lambda_read(value):
    if pd.isna(value) or abs(float(value)) < 1e-12:
        return "neutral"
    if value < 0:
        return "reverse/noisy residual"
    return "same-direction correction"


RESIDUAL_SPECS = [
    {
        "name": "R01_driver_logit_bias",
        "label": "Driver logit residual",
        "groups": {"Driver": 1.00},
        "alphas": {"Driver": 240},
    },
    {
        "name": "R02_driver_phase_logit_bias",
        "label": "Driver + phase residual",
        "groups": {"Driver": 1.00, "Driver_Phase": 0.65},
        "alphas": {"Driver": 260, "Driver_Phase": 420},
    },
    {
        "name": "R03_driver_compound_logit_bias",
        "label": "Driver + compound residual",
        "groups": {"Driver": 1.00, "Driver_Compound": 0.55},
        "alphas": {"Driver": 260, "Driver_Compound": 520},
    },
    {
        "name": "R04_driver_stint_logit_bias",
        "label": "Driver + stint residual",
        "groups": {"Driver": 1.00, "Driver_Stint": 0.50},
        "alphas": {"Driver": 260, "Driver_Stint": 520},
    },
    {
        "name": "R05_driver_strategy_logit_bias",
        "label": "Driver strategy residual bundle",
        "groups": {"Driver": 1.00, "Driver_Phase": 0.50, "Driver_Compound": 0.40, "Driver_Stint": 0.35},
        "alphas": {"Driver": 300, "Driver_Phase": 520, "Driver_Compound": 650, "Driver_Stint": 650},
    },
    {
        "name": "R06_driver_race_logit_bias",
        "label": "Driver + race residual stress test",
        "groups": {"Driver": 1.00, "Driver_Race": 0.20},
        "alphas": {"Driver": 320, "Driver_Race": 1200},
    },
]


def evaluate_logit_residual_spec(spec, train_resid_frame, test_resid_frame, y, base_oof, base_test):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE + 100)
    delta_oof = np.zeros(len(train_resid_frame), dtype=float)
    delta_test = np.zeros(len(test_resid_frame), dtype=float)
    fold_signal_rows = []

    for fold, (fit_idx, valid_idx) in enumerate(skf.split(train_resid_frame, y), start=1):
        maps = fit_logit_residual_maps(
            train_resid_frame.iloc[fit_idx],
            y[fit_idx],
            base_oof[fit_idx],
            spec["alphas"],
        )
        valid_delta = apply_logit_residual_signal(train_resid_frame.iloc[valid_idx], maps, spec["groups"])
        test_delta = apply_logit_residual_signal(test_resid_frame, maps, spec["groups"])
        delta_oof[valid_idx] = valid_delta
        delta_test += test_delta / N_SPLITS
        fold_signal_rows.append({
            "Experiment": spec["name"],
            "Fold": fold,
            "Delta_Logit_Mean": float(np.mean(valid_delta)),
            "Delta_Logit_Std": float(np.std(valid_delta)),
            "Delta_Logit_Min": float(np.min(valid_delta)),
            "Delta_Logit_Max": float(np.max(valid_delta)),
        })

    base_logit_oof = logit_clip(base_oof)
    base_logit_test = logit_clip(base_test)
    base_auc = roc_auc_score(y, base_oof)
    lambda_grid = np.array([-1.00, -0.75, -0.50, -0.25, -0.10, 0.0, 0.10, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50])
    rows = []

    for lam in lambda_grid:
        corrected_oof = sigmoid_array(base_logit_oof + lam * delta_oof)
        auc = roc_auc_score(y, corrected_oof)
        rows.append({
            "Experiment": spec["name"],
            "Label": spec["label"],
            "Lambda": lam,
            "OOF_AUC": auc,
            "Delta_vs_Base": auc - base_auc,
        })

    strength_table = pd.DataFrame(rows)
    best_row = strength_table.loc[strength_table["OOF_AUC"].idxmax()].copy()
    best_lambda = float(best_row["Lambda"])
    corrected_oof = sigmoid_array(base_logit_oof + best_lambda * delta_oof)
    corrected_test = sigmoid_array(base_logit_test + best_lambda * delta_test)

    fold_rows = []
    for fold, (_, valid_idx) in enumerate(skf.split(train_resid_frame, y), start=1):
        base_fold_auc = roc_auc_score(y[valid_idx], base_oof[valid_idx])
        corrected_fold_auc = roc_auc_score(y[valid_idx], corrected_oof[valid_idx])
        fold_rows.append({
            "Experiment": spec["name"],
            "Fold": fold,
            "Base_AUC": base_fold_auc,
            "Correction_AUC": corrected_fold_auc,
            "Delta": corrected_fold_auc - base_fold_auc,
            "Lambda": best_lambda,
        })

    result = {
        "spec": spec,
        "oof": corrected_oof,
        "test_pred": corrected_test,
        "auc": roc_auc_score(y, corrected_oof),
        "logloss": log_loss(y, np.clip(corrected_oof, 1e-6, 1 - 1e-6)),
        "folds": pd.DataFrame(fold_rows),
        "strength_table": strength_table,
        "signal_summary": pd.DataFrame(fold_signal_rows),
        "best_lambda": best_lambda,
        "delta_vs_base": roc_auc_score(y, corrected_oof) - base_auc,
    }
    return result


def run_driver_residual_correction(base_result):
    y = cv_train[TARGET].astype(int).to_numpy()
    base_oof = np.clip(base_result["oof"], 1e-6, 1 - 1e-6)
    base_test = np.clip(base_result["test_pred"], 1e-6, 1 - 1e-6)
    train_resid_frame = build_residual_frame(cv_train)
    test_resid_frame = build_residual_frame(test)

    spec_results = [
        evaluate_logit_residual_spec(spec, train_resid_frame, test_resid_frame, y, base_oof, base_test)
        for spec in RESIDUAL_SPECS
    ]
    base_auc = roc_auc_score(y, base_oof)

    candidate_rows = [{
        "Experiment": "R00_base_selected_stack",
        "Label": base_result["config"].label,
        "Groups": "none",
        "Lambda": 0.0,
        "OOF_AUC": base_auc,
        "Delta_vs_Base": 0.0,
        "Decision": "base",
        "Lambda_Read": "none",
    }]
    for result in spec_results:
        spec = result["spec"]
        delta = result["delta_vs_base"]
        if delta >= RESIDUAL_PROMOTION_MIN_AUC:
            decision = "promote"
        elif delta > 0:
            decision = "watchlist"
        else:
            decision = "reject"
        candidate_rows.append({
            "Experiment": spec["name"],
            "Label": spec["label"],
            "Groups": " + ".join(spec["groups"].keys()),
            "Lambda": result["best_lambda"],
            "OOF_AUC": result["auc"],
            "Delta_vs_Base": delta,
            "Decision": decision,
            "Lambda_Read": residual_lambda_read(result["best_lambda"]),
        })
    candidate_table = pd.DataFrame(candidate_rows)

    best_result = max(spec_results, key=lambda item: item["auc"])
    if best_result["auc"] < base_auc + RESIDUAL_PROMOTION_MIN_AUC:
        corrected_oof = base_oof
        corrected_test = base_test
        selected_name = "R00_base_selected_stack"
        selected_label = "No residual correction"
        selected_auc = base_auc
        selected_logloss = log_loss(y, base_oof)
        selected_folds = best_result["folds"].copy()
        selected_signal = pd.concat([r["signal_summary"] for r in spec_results], ignore_index=True)
        selected_strength = pd.concat([r["strength_table"] for r in spec_results], ignore_index=True)
    else:
        corrected_oof = best_result["oof"]
        corrected_test = best_result["test_pred"]
        selected_name = best_result["spec"]["name"]
        selected_label = best_result["spec"]["label"]
        selected_auc = best_result["auc"]
        selected_logloss = best_result["logloss"]
        selected_folds = best_result["folds"]
        selected_signal = best_result["signal_summary"]
        selected_strength = best_result["strength_table"]

    return {
        "config": ExperimentConfig(
            selected_name,
            selected_label,
            base_result["config"].blocks,
            include_driver_ohe=base_result["config"].include_driver_ohe,
            model_kind="driver_logit_residual",
            original_weight=base_result["config"].original_weight,
        ),
        "oof": corrected_oof,
        "test_pred": corrected_test,
        "folds": selected_folds,
        "auc": selected_auc,
        "logloss": selected_logloss,
        "models": [],
        "numeric_features": [],
        "categorical_features": [],
        "candidate_table": candidate_table,
        "strength_table": selected_strength,
        "signal_summary": selected_signal,
        "base_auc": base_auc,
    }


auxiliary_results = []
if RUN_RESIDUAL_CORRECTION:
    residual_correction_result = run_driver_residual_correction(best_ladder_result)
    residual_delta = residual_correction_result["auc"] - best_ladder_result["auc"]
    residual_promoted = residual_delta >= RESIDUAL_PROMOTION_MIN_AUC

    display_table(
        residual_correction_result["candidate_table"],
        "Driver logit residual correction: candidate groups",
    )
    display_table(
        residual_correction_result["signal_summary"].head(18),
        "Driver logit residual correction: fold-level signal scale",
    )
    display_table(
        residual_correction_result["strength_table"].sort_values(["Experiment", "OOF_AUC"], ascending=[True, False]).groupby("Experiment").head(4),
        "Driver logit residual correction: best lambda search rows",
    )

    if residual_promoted:
        display_cards([
            metric_card("Residual layer", "promote", f"+{residual_delta:.5f} AUC", COLORS["green"]),
            metric_card("Driver role", "logit correction", "base stack stays the engine", COLORS["blue"]),
        ], columns=2)
        auxiliary_results.append(residual_correction_result)
    else:
        display_cards([
            metric_card("Residual layer", "watchlist" if residual_delta > 0 else "not promoted", f"{residual_delta:+.6f} AUC", COLORS["gold"] if residual_delta > 0 else COLORS["red"]),
            metric_card("Driver role", "candidate", "retry on heavy stack", COLORS["gold"]),
        ], columns=2)
else:
    residual_correction_result = None
    auxiliary_results = []
    print("Driver residual correction skipped.")


## 11. Heavy Stack Residual Overlay

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-left:6px solid #3D6F8A; border-radius:14px; padding:13px 15px; margin:10px 0 16px 0; color:#233127; line-height:1.58; box-shadow:0 8px 18px rgba(35,49,39,0.04);">
  <div style="font-size:13px; font-weight:900; letter-spacing:.08em; text-transform:uppercase; color:#3D6F8A; margin-bottom:7px;">Driver +alpha on the real scoring base</div>
  <div style="font-size:1.08rem; font-weight:900; color:#233127; margin-bottom:10px;">Kaggle submission path: attach the full stack output, then let this notebook test whether Driver residuals add anything on top.</div>
  <table style="width:100%; border-collapse:collapse; font-size:13.5px; line-height:1.45;">
    <thead><tr style="background:#233127; color:#FFFCF8; text-align:left;"><th style="padding:9px 10px; border:1px solid #D9D0BF;">Layer</th><th style="padding:9px 10px; border:1px solid #D9D0BF;">Role</th><th style="padding:9px 10px; border:1px solid #D9D0BF;">Decision rule</th></tr></thead>
    <tbody>
      <tr><td style="padding:9px 10px; border:1px solid #E2D8C7; font-weight:900; color:#2F6B4F;">Full prediction artifacts</td><td style="padding:9px 10px; border:1px solid #E2D8C7;">Read tagged <b>full</b> OOF/test predictions from attached stack and RealMLP outputs.</td><td style="padding:9px 10px; border:1px solid #E2D8C7;">Default paths are <b>/kaggle/input/notebooks/pilkwang/s6e5-high-score-stacking</b> and <b>/kaggle/input/notebooks/pilkwang/s6e5-realmlp-oof-artifact-builder</b>.</td></tr>
      <tr style="background:#F7F3EA;"><td style="padding:9px 10px; border:1px solid #E2D8C7; font-weight:900; color:#C7772C;">Blend base</td><td style="padding:9px 10px; border:1px solid #E2D8C7;">Search probability and rank blends, following the blend notebook.</td><td style="padding:9px 10px; border:1px solid #E2D8C7;">Rank can win scoring; probability is the cleaner base for logit correction.</td></tr>
      <tr><td style="padding:9px 10px; border:1px solid #E2D8C7; font-weight:900; color:#B5473C;">Driver residual</td><td style="padding:9px 10px; border:1px solid #E2D8C7;">Add fold-safe Driver group bias to the heavy-stack logit.</td><td style="padding:9px 10px; border:1px solid #E2D8C7;">Promote only if it beats the best artifact base, not merely the probability base.</td></tr>
    </tbody>
  </table>
</div>

In [ ]:
def stack_artifact_roots():
    candidates = []
    for explicit_path in [STACKING_INPUT_PATH, REALMLP_INPUT_PATH, *EXTRA_ARTIFACT_INPUTS]:
        if explicit_path.exists():
            candidates.append(explicit_path)
    candidates.append(Path("."))
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates.extend([p for p in kaggle_root.iterdir() if p.is_dir()])
    roots = []
    seen = set()
    for root in candidates:
        key = str(root.resolve()) if root.exists() else str(root)
        if key not in seen:
            roots.append(root)
            seen.add(key)
    return roots


def stack_model_name_from_path(path, prefix):
    name = path.stem
    return name[len(prefix):] if name.startswith(prefix) else name


def discover_stack_artifacts(profile):
    oof_paths = {}
    sub_paths = {}
    roots = stack_artifact_roots()
    for root in roots:
        for oof_path in root.glob("oof_*.csv"):
            if profile and profile != "any" and f"_{profile}_" not in oof_path.stem:
                continue
            oof_paths.setdefault(stack_model_name_from_path(oof_path, "oof_"), oof_path)
        for sub_path in root.glob("sub_*.csv"):
            if profile and profile != "any" and f"_{profile}_" not in sub_path.stem:
                continue
            sub_paths.setdefault(stack_model_name_from_path(sub_path, "sub_"), sub_path)
    names = sorted(set(oof_paths).intersection(sub_paths))
    manifest = pd.DataFrame({
        "Model": names,
        "OOF_File": [str(oof_paths[name]) for name in names],
        "Submission_File": [str(sub_paths[name]) for name in names],
    })
    return names, oof_paths, sub_paths, roots, manifest


def load_stack_prediction_frames(names, oof_paths, sub_paths):
    oof_frames = []
    sub_frames = []
    row_checks = []
    for name in names:
        oof = pd.read_csv(oof_paths[name])
        sub = pd.read_csv(sub_paths[name])
        row_checks.append({"Model": name, "OOF_Rows": len(oof), "Submission_Rows": len(sub)})
        pred_col = "pred" if "pred" in oof.columns else next(c for c in TARGET_CANDIDATES if c in oof.columns)
        target_col = "target" if "target" in oof.columns else next(c for c in TARGET_CANDIDATES if c in oof.columns and c != pred_col)
        sub_pred_col = next((c for c in TARGET_CANDIDATES if c in sub.columns), None)
        if sub_pred_col is None and "pred" in sub.columns:
            sub_pred_col = "pred"
        if sub_pred_col is None:
            raise ValueError(f"No prediction column found in {sub_paths[name]}")
        oof_frames.append(oof[[ID_COL, target_col, pred_col]].rename(columns={target_col: "target", pred_col: name}))
        sub_frames.append(sub[[ID_COL, sub_pred_col]].rename(columns={sub_pred_col: name}))

    oof_blend_frame = oof_frames[0]
    for frame in oof_frames[1:]:
        oof_blend_frame = oof_blend_frame.merge(frame, on=[ID_COL, "target"], how="inner")
    sub_blend_frame = sub_frames[0]
    for frame in sub_frames[1:]:
        sub_blend_frame = sub_blend_frame.merge(frame, on=ID_COL, how="inner")

    row_checks = pd.DataFrame(row_checks)
    if row_checks["OOF_Rows"].nunique() != 1:
        raise ValueError("Stack OOF artifacts have different row counts.")
    if row_checks["Submission_Rows"].nunique() != 1:
        raise ValueError("Stack submission artifacts have different row counts.")
    if len(oof_blend_frame) != int(row_checks["OOF_Rows"].iloc[0]):
        raise ValueError("Stack OOF artifacts do not align on id and target.")
    if len(sub_blend_frame) != int(row_checks["Submission_Rows"].iloc[0]):
        raise ValueError("Stack submission artifacts do not align on id.")
    return oof_blend_frame, sub_blend_frame, row_checks


def align_stack_predictions_to_driver_context(oof_blend_frame, sub_blend_frame, names):
    cv_index = cv_train[[ID_COL, TARGET]].reset_index().rename(columns={TARGET: "target_cv"})
    aligned_oof = cv_index.merge(oof_blend_frame, on=ID_COL, how="inner")
    if "target" in aligned_oof.columns:
        mismatch = aligned_oof["target"].astype(int).to_numpy() != aligned_oof["target_cv"].astype(int).to_numpy()
        if mismatch.any():
            raise ValueError("Stack OOF targets do not match the Driver notebook target on aligned IDs.")
    aligned_oof = aligned_oof.sort_values("index").reset_index(drop=True)
    stack_train_frame = cv_train.iloc[aligned_oof["index"].to_numpy()].reset_index(drop=True)
    stack_y = stack_train_frame[TARGET].astype(int).to_numpy()
    stack_oof_matrix = aligned_oof[names].to_numpy(dtype=float)

    test_index = test[[ID_COL]].reset_index()
    aligned_test = test_index.merge(sub_blend_frame, on=ID_COL, how="left").sort_values("index").reset_index(drop=True)
    if aligned_test[names].isna().any().any():
        raise ValueError("Stack submission artifacts are missing one or more test IDs.")
    stack_test_frame = test.iloc[aligned_test["index"].to_numpy()].reset_index(drop=True)
    stack_test_matrix = aligned_test[names].to_numpy(dtype=float)
    can_blend_with_current = len(stack_train_frame) == len(cv_train) and np.array_equal(
        stack_train_frame[ID_COL].to_numpy(), cv_train[ID_COL].to_numpy()
    )
    return stack_train_frame, stack_test_frame, stack_y, stack_oof_matrix, stack_test_matrix, can_blend_with_current


def simplex_grid(n, step):
    units = int(round(1 / step))
    current = []
    def rec(left, slots):
        if slots == 1:
            yield current + [left]
        else:
            for value in range(left + 1):
                current.append(value)
                yield from rec(left - value, slots - 1)
                current.pop()
    for parts in rec(units, n):
        yield np.array(parts, dtype=float) / units


def rank01_np(values):
    values = np.asarray(values, dtype=float)
    order = np.argsort(values, kind="mergesort")
    ranks = np.empty(len(values), dtype=float)
    ranks[order] = np.arange(1, len(values) + 1)
    return ranks / (len(values) + 1)


def search_stack_base_blends(names, oof_matrix, test_matrix, y):
    rank_oof = np.column_stack([rank01_np(oof_matrix[:, idx]) for idx in range(oof_matrix.shape[1])])
    rank_test = np.column_stack([rank01_np(test_matrix[:, idx]) for idx in range(test_matrix.shape[1])])
    candidates = []

    if len(names) == 1:
        weights = np.array([1.0])
        candidates.append({
            "Mode": "single probability model",
            "OOF_AUC": roc_auc_score(y, oof_matrix[:, 0]),
            "Weights": weights,
            "OOF": oof_matrix[:, 0],
            "Test": test_matrix[:, 0],
        })
    else:
        for mode, matrix_oof, matrix_test in [
            ("probability", oof_matrix, test_matrix),
            ("rank", rank_oof, rank_test),
        ]:
            best_auc = -np.inf
            best_weights = None
            best_oof = None
            best_test = None
            for weights in simplex_grid(len(names), STACK_BLEND_GRID_STEP):
                pred = matrix_oof @ weights
                auc = roc_auc_score(y, pred)
                if auc > best_auc:
                    best_auc = auc
                    best_weights = weights.copy()
                    best_oof = pred
                    best_test = matrix_test @ weights
            candidates.append({
                "Mode": mode,
                "OOF_AUC": best_auc,
                "Weights": best_weights,
                "OOF": best_oof,
                "Test": best_test,
            })

    base_scores = pd.DataFrame([
        {"Model": name, "OOF_AUC": roc_auc_score(y, oof_matrix[:, idx])}
        for idx, name in enumerate(names)
    ]).sort_values("OOF_AUC", ascending=False)
    blend_rows = []
    for candidate in candidates:
        for name, weight in zip(names, candidate["Weights"]):
            if weight > 0:
                blend_rows.append({"Blend_Mode": candidate["Mode"], "Model": name, "Weight": weight, "Blend_OOF_AUC": candidate["OOF_AUC"]})
    blend_table = pd.DataFrame(blend_rows).sort_values(["Blend_OOF_AUC", "Blend_Mode", "Weight"], ascending=[False, True, False])
    best_candidate = max(candidates, key=lambda item: item["OOF_AUC"])
    probability_candidate = next(item for item in candidates if item["Mode"] in ["probability", "single probability model"])
    return base_scores, blend_table, best_candidate, probability_candidate


def run_heavy_stack_residual_overlay(stack_train_frame, stack_test_frame, y, probability_base, best_artifact_base):
    train_resid_frame = build_residual_frame(stack_train_frame)
    test_resid_frame = build_residual_frame(stack_test_frame)
    spec_results = [
        evaluate_logit_residual_spec(
            spec,
            train_resid_frame,
            test_resid_frame,
            y,
            np.clip(probability_base["OOF"], 1e-6, 1 - 1e-6),
            np.clip(probability_base["Test"], 1e-6, 1 - 1e-6),
        )
        for spec in RESIDUAL_SPECS
    ]
    rows = [{
        "Experiment": "H00_best_stack_artifact",
        "Label": best_artifact_base["Mode"],
        "Groups": "none",
        "Lambda": 0.0,
        "OOF_AUC": best_artifact_base["OOF_AUC"],
        "Delta_vs_Probability_Base": best_artifact_base["OOF_AUC"] - probability_base["OOF_AUC"],
        "Delta_vs_Best_Artifact": 0.0,
        "Decision": "base",
        "Lambda_Read": "none",
    }, {
        "Experiment": "H00_probability_stack_base",
        "Label": probability_base["Mode"],
        "Groups": "none",
        "Lambda": 0.0,
        "OOF_AUC": probability_base["OOF_AUC"],
        "Delta_vs_Probability_Base": 0.0,
        "Delta_vs_Best_Artifact": probability_base["OOF_AUC"] - best_artifact_base["OOF_AUC"],
        "Decision": "probability base",
        "Lambda_Read": "none",
    }]
    for result in spec_results:
        spec = result["spec"]
        delta_best = result["auc"] - best_artifact_base["OOF_AUC"]
        delta_prob = result["auc"] - probability_base["OOF_AUC"]
        if delta_best >= RESIDUAL_PROMOTION_MIN_AUC:
            decision = "promote"
        elif delta_prob > 0:
            decision = "watchlist"
        else:
            decision = "reject"
        rows.append({
            "Experiment": "H_" + spec["name"],
            "Label": spec["label"],
            "Groups": " + ".join(spec["groups"].keys()),
            "Lambda": result["best_lambda"],
            "OOF_AUC": result["auc"],
            "Delta_vs_Probability_Base": delta_prob,
            "Delta_vs_Best_Artifact": delta_best,
            "Decision": decision,
            "Lambda_Read": residual_lambda_read(result["best_lambda"]),
        })
    candidate_table = pd.DataFrame(rows).sort_values("OOF_AUC", ascending=False)
    best_residual = max(spec_results, key=lambda item: item["auc"])
    if best_residual["auc"] >= best_artifact_base["OOF_AUC"] + RESIDUAL_PROMOTION_MIN_AUC:
        selected_oof = best_residual["oof"]
        selected_test = best_residual["test_pred"]
        selected_name = "H_" + best_residual["spec"]["name"]
        selected_label = "Heavy stack + " + best_residual["spec"]["label"]
        selected_auc = best_residual["auc"]
        selected_logloss = best_residual["logloss"]
        selected_decision = "promoted residual"
    else:
        selected_oof = best_artifact_base["OOF"]
        selected_test = best_artifact_base["Test"]
        selected_name = "H00_heavy_stack_artifact"
        selected_label = f"Heavy stack artifact ({best_artifact_base['Mode']})"
        selected_auc = best_artifact_base["OOF_AUC"]
        selected_logloss = log_loss(y, np.clip(selected_oof, 1e-6, 1 - 1e-6))
        selected_decision = "artifact base"

    selected_result = {
        "config": ExperimentConfig(
            selected_name,
            selected_label,
            blocks=[],
            include_driver_ohe=False,
            model_kind="artifact_stack",
            original_weight=0.0,
        ),
        "oof": selected_oof,
        "test_pred": selected_test,
        "folds": pd.DataFrame(),
        "auc": selected_auc,
        "logloss": selected_logloss,
        "models": [],
        "numeric_features": [],
        "categorical_features": [],
        "candidate_table": candidate_table,
        "decision": selected_decision,
    }
    return selected_result, candidate_table, spec_results


heavy_stack_result = None
heavy_stack_candidate_table = pd.DataFrame()
if RUN_HEAVY_STACK_RESIDUAL:
    try:
        stack_names, stack_oof_paths, stack_sub_paths, stack_roots, stack_manifest = discover_stack_artifacts(STACK_BLEND_PROFILE)
        display_table(pd.DataFrame({"Search_Order": range(1, len(stack_roots) + 1), "Root": [str(root) for root in stack_roots[:12]]}), "Heavy-stack artifact search roots", max_rows=None)
        display_table(stack_manifest, f"Discovered {STACK_BLEND_PROFILE} stack artifacts", max_rows=None)
        if not stack_names:
            print("No paired stack artifacts found for the selected profile.")
        else:
            stack_oof_frame, stack_sub_frame, stack_row_checks = load_stack_prediction_frames(stack_names, stack_oof_paths, stack_sub_paths)
            display_table(stack_row_checks, "Heavy-stack artifact row-count check", precision=0, max_rows=None)
            stack_train_frame, stack_test_frame, stack_y, stack_oof_matrix, stack_test_matrix, can_blend_with_current = align_stack_predictions_to_driver_context(
                stack_oof_frame,
                stack_sub_frame,
                stack_names,
            )
            stack_base_scores, stack_blend_table, stack_best_base, stack_probability_base = search_stack_base_blends(
                stack_names,
                stack_oof_matrix,
                stack_test_matrix,
                stack_y,
            )
            display_table(stack_base_scores, "Heavy-stack artifact base OOF scores", max_rows=None)
            display_table(stack_blend_table, "Heavy-stack blend candidates", max_rows=None)
            heavy_stack_result, heavy_stack_candidate_table, heavy_stack_spec_results = run_heavy_stack_residual_overlay(
                stack_train_frame,
                stack_test_frame,
                stack_y,
                stack_probability_base,
                stack_best_base,
            )
            display_table(heavy_stack_candidate_table, "Heavy-stack Driver residual overlay candidates", max_rows=None)

            cards = [
                metric_card("Stack rows", f"{len(stack_train_frame):,}", "OOF rows aligned to Driver notebook", COLORS["blue"]),
                metric_card("Best artifact", f"{stack_best_base['OOF_AUC']:.6f}", stack_best_base["Mode"], COLORS["green"]),
                metric_card("Selected overlay", f"{heavy_stack_result['auc']:.6f}", heavy_stack_result["decision"], COLORS["gold"]),
                metric_card("Blend eligible", "yes" if can_blend_with_current else "no", "requires identical cv_train IDs", COLORS["green"] if can_blend_with_current else COLORS["red"]),
            ]
            display_cards(cards, columns=4)
            if can_blend_with_current:
                if "auxiliary_results" not in globals():
                    auxiliary_results = []
                auxiliary_results.append(heavy_stack_result)
            else:
                print("Heavy-stack overlay is reported but not added to the final blend because OOF IDs do not exactly match cv_train.")
    except Exception as exc:
        print(f"Heavy-stack residual overlay skipped: {type(exc).__name__}: {exc}")
else:
    print("Heavy-stack residual overlay skipped.")

## 12. Original Dataset Weight Ablation


In [ ]:
if RUN_ORIGINAL_ABLATION and original is not None:
    original_configs = [
        ExperimentConfig(
            name=f"O{str(weight).replace('.', '_')}_orig_weight",
            label=f"Original weight {weight}",
            blocks=best_ladder_config.blocks,
            include_driver_ohe=best_ladder_config.include_driver_ohe,
            model_kind=LADDER_MODEL,
            original_weight=weight,
        )
        for weight in ORIGINAL_WEIGHTS
    ]
    original_results = [run_oof_experiment(cfg, verbose=True) for cfg in original_configs]
    original_summary = summarize_results(original_results, baseline_auc=best_ladder_result["auc"])
    display_table(original_summary, "Original dataset weight ablation")
    plot_ladder(original_summary, "Original data weight ablation")
    best_original_idx = int(original_summary["OOF_AUC"].idxmax())
    selected_feature_result = original_results[best_original_idx]
else:
    print("Original dataset ablation skipped.")
    original_results = []
    original_summary = pd.DataFrame()
    selected_feature_result = best_ladder_result

selected_config = selected_feature_result["config"]
print(f"Selected feature/data setup: {selected_config.name} | OOF AUC {selected_feature_result['auc']:.5f}")

## 13. Model Upgrade and OOF Blend

<div style="background:#FFFCF8; border:1px solid #E2D8C7; border-left:6px solid #B5473C; border-radius:14px; padding:13px 15px; margin:10px 0 16px 0; color:#233127; line-height:1.58; box-shadow:0 8px 18px rgba(35,49,39,0.04);">The selected Driver model only becomes a leaderboard candidate if it earns blend weight beside heavier GBDT models. OOF rank blending is the decision rule.</div>


In [ ]:
model_results = []
if RUN_MODEL_UPGRADE:
    for model_name in FINAL_MODEL_NAMES:
        try:
            model_label = "CatBoost on preprocessed/OHE features" if model_name == "cat" else model_name.upper()
            cfg = ExperimentConfig(
                name=f"M_{model_name}",
                label=f"Best features + {model_label}",
                blocks=selected_config.blocks,
                include_driver_ohe=selected_config.include_driver_ohe,
                model_kind=model_name,
                original_weight=selected_config.original_weight,
            )
            model_results.append(run_oof_experiment(cfg, verbose=True))
        except Exception as exc:
            print(f"Skipped {model_name}: {type(exc).__name__}: {exc}")
else:
    model_results = [selected_feature_result]

if not model_results:
    model_results = [selected_feature_result]

if "auxiliary_results" in globals() and auxiliary_results:
    model_results = model_results + auxiliary_results

model_summary = summarize_results(model_results, baseline_auc=selected_feature_result["auc"])
display_table(model_summary, "Model upgrade results")
plot_ladder(model_summary, "Model upgrade OOF AUC")

blend_names = [result["config"].name for result in model_results]
blend_oofs = np.column_stack([result["oof"] for result in model_results])
blend_tests = np.column_stack([result["test_pred"] for result in model_results])
y_cv = cv_train[TARGET].astype(int).to_numpy()


def rank01(values):
    values = np.asarray(values, dtype=float)
    order = np.argsort(values, kind="mergesort")
    ranks = np.empty(len(values), dtype=float)
    ranks[order] = np.arange(1, len(values) + 1)
    return ranks / len(values)

if len(model_results) == 1:
    blend_weights = np.array([1.0])
    blend_oof = blend_oofs[:, 0]
    blend_test = blend_tests[:, 0]
    blend_auc = roc_auc_score(y_cv, blend_oof)
    blend_method = "single model"
else:
    step = 0.10 if FAST_MODE else 0.05
    best_prob = (-np.inf, None, None)
    for weights in simplex_grid(len(model_results), step):
        pred = blend_oofs @ weights
        auc = roc_auc_score(y_cv, pred)
        if auc > best_prob[0]:
            best_prob = (auc, weights, pred)

    rank_oofs = np.column_stack([rank01(blend_oofs[:, idx]) for idx in range(blend_oofs.shape[1])])
    rank_tests = np.column_stack([rank01(blend_tests[:, idx]) for idx in range(blend_tests.shape[1])])
    rank_blend_oof = rank_oofs.mean(axis=1)
    rank_blend_test = rank_tests.mean(axis=1)
    rank_blend_auc = roc_auc_score(y_cv, rank_blend_oof)

    simple_blend_oof = blend_oofs.mean(axis=1)
    simple_blend_test = blend_tests.mean(axis=1)
    simple_blend_auc = roc_auc_score(y_cv, simple_blend_oof)

    candidates = [
        (best_prob[0], "OOF probability weight search", best_prob[1], best_prob[2], blend_tests @ best_prob[1]),
        (rank_blend_auc, "equal rank average", np.full(len(model_results), 1 / len(model_results)), rank_blend_oof, rank_blend_test),
        (simple_blend_auc, "equal probability average", np.full(len(model_results), 1 / len(model_results)), simple_blend_oof, simple_blend_test),
    ]
    blend_auc, blend_method, blend_weights, blend_oof, blend_test = max(candidates, key=lambda item: item[0])

blend_table = pd.DataFrame({"Model": blend_names, "Blend_Weight": blend_weights})
display_table(blend_table, f"Selected blend: {blend_method} | OOF AUC {blend_auc:.5f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.kdeplot(blend_oof, ax=axes[0], color=COLORS["green"], fill=True, alpha=0.25)
axes[0].axvline(cv_train[TARGET].mean(), color=COLORS["red"], linestyle="--", label="base rate")
axes[0].set_title("OOF probability distribution", fontweight="bold")
axes[0].legend()
sns.kdeplot(blend_test, ax=axes[1], color=COLORS["blue"], fill=True, alpha=0.25)
axes[1].axvline(blend_test.mean(), color=COLORS["gold"], linestyle="--", label="test mean")
axes[1].set_title("Test probability distribution", fontweight="bold")
axes[1].legend()
plt.tight_layout()
plt.show()


## 14. Final Submission


In [ ]:
submission = submission_template.copy()
submission[SUBMISSION_TARGET] = np.clip(blend_test, 0, 1)

assert submission.shape[0] == test.shape[0]
assert submission[ID_COL].equals(test[ID_COL])
assert submission[SUBMISSION_TARGET].between(0, 1).all()
assert submission[SUBMISSION_TARGET].isna().sum() == 0

submission.to_csv(SUBMISSION_FILENAME, index=False)

submission_summary = pd.DataFrame({
    "Rows": [len(submission)],
    "Mean_Prediction": [submission[SUBMISSION_TARGET].mean()],
    "Min_Prediction": [submission[SUBMISSION_TARGET].min()],
    "Max_Prediction": [submission[SUBMISSION_TARGET].max()],
    "Filename": [SUBMISSION_FILENAME],
})
display_table(submission_summary, "Submission sanity check")
display_table(submission.head(10), "Submission preview")